# Training & Evaluation — Patch + Cross-Modal Differential Features (CMDF)

**Pipeline Version: v3 — Three-Way Comparison**

Run this notebook AFTER `feature_extraction_cmdf.ipynb` has completed.

**Three pipelines compared:**

| Pipeline | Input Features | Models |
|---|---|---|
| **Baseline** | Whole-tumour T1Gd+T2 radiomics (~214 features) | ECNN, KNN, DT, GNB, RF |
| **Patch** | 16³ patch-agg stats (mean/std/min/max/median) | ECNN, KNN, DT, GNB, RF, MIL |
| **Patch+CMDF** | Patch-agg + cross-modal differential features | ECNN, KNN, DT, GNB, RF, MIL |

All pipelines use identical 5-fold stratified CV for fair comparison.

## 1. Install & Mount

In [ ]:
!pip install pandas scikit-learn tensorflow matplotlib seaborn tqdm pdfplumber scipy -q
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 61.8 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/Capstone_paper_001', exist_ok=True)
%cd /content/drive/MyDrive/Capstone_paper_001

/content/drive/MyDrive/Capstone_paper_001


## 2. config.py

In [ ]:
%%writefile config.py
"""
config.py – Centralized configuration for the 1p/19q co-deletion prediction pipeline.
Version 3: Patch-Based + Cross-Modal Differential Features (CMDF)
"""

import os

# ──────────────────────────────────────────────────────────────────────────────
#  PATHS  —  UTSW Glioma dataset (per-subject folders + TSV metadata)
# ──────────────────────────────────────────────────────────────────────────────
DATA_ROOT = "/content/drive/MyDrive/UTSW/UTSW-Glioma"          # per-subject imaging folders (e.g. BT0001)
METADATA_TSV_PATH = "/content/drive/MyDrive/UTSW/UTSW_Glioma_Metadata-2-1.tsv"

IMAGES_ROOT_DIR = DATA_ROOT
PATIENT_RAW_DIR = DATA_ROOT     # kept for backward-compat with segmentation.py / patch_extraction.py

# Each sequence maps to a list of candidate subject-folder-relative filenames,
# tried in order (primary file first, ANTs-registered fallback second) — matches
# this dataset's on-disk naming convention.
IMAGE_PATH_TEMPLATES = {
    "t1Gd": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1ce.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1ce_ants.nii.gz"),
    ],
    "t2": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t2.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t2_ants.nii.gz"),
    ],
    # Not consumed by the CMDF pipeline (which only compares t1Gd vs t2), but
    # defined here in case SEQUENCES is extended later.
    "t1": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1_ants.nii.gz"),
    ],
    "flair": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_flair.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_fl_ants.nii.gz"),
    ],
}

# Segmentation mask candidates, tried in order, relative to each subject folder.
SEG_PATH_TEMPLATES = [
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "rtumorseg_manual_correction.nii.gz"),
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "tumorseg_manual_correction.nii.gz"),
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "tumorseg_FeTS.nii.gz"),
]

PROJECT_ROOT = "/content/drive/MyDrive/Capstone_paper_001"
OUTPUT_ROOT  = os.path.join(PROJECT_ROOT, "output")

PREPROCESS_DIR   = os.path.join(OUTPUT_ROOT, "preprocessed")
FEATURES_DIR     = os.path.join(OUTPUT_ROOT, "features")
AUGMENTED_DIR    = os.path.join(OUTPUT_ROOT, "augmented")
RESULTS_DIR      = os.path.join(OUTPUT_ROOT, "results")
MODELS_DIR       = os.path.join(OUTPUT_ROOT, "models")

# ──────────────────────────────────────────────────────────────────────────────
#  SEQUENCES
# ──────────────────────────────────────────────────────────────────────────────
# CMDF compares exactly these two modalities against each other. T1 / FLAIR are
# defined above in IMAGE_PATH_TEMPLATES but unused unless you extend the pipeline.
SEQUENCES = ["t1Gd", "t2"]

# ──────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ──────────────────────────────────────────────────────────────────────────────
TARGET_SPACING = (1.0, 1.0, 1.0)
N4_ITERATIONS  = [50, 50, 30, 20]
N4_CONVERGENCE_THRESHOLD = 1e-6
N4_SHRINK_FACTOR = 3

# ──────────────────────────────────────────────────────────────────────────────
#  PYRADIOMICS (shared by whole-tumour & patch extractors)
# ──────────────────────────────────────────────────────────────────────────────
PYRADIOMICS_PARAMS = {
    "binWidth": 25,
    "resampledPixelSpacing": None,
    "interpolator": "sitkBSpline",
    "normalizeScale": 1,
    "normalize": True,
    "removeOutliers": 3,
}

FEATURE_CLASSES = [
    "firstorder", "shape", "glcm", "glszm", "glrlm", "ngtdm", "gldm",
]

# ──────────────────────────────────────────────────────────────────────────────
#  VAE DATA AUGMENTATION
# ──────────────────────────────────────────────────────────────────────────────
VAE_LATENT_DIM    = 32
VAE_HIDDEN_UNITS  = [128, 64]
VAE_EPOCHS        = 200
VAE_BATCH_SIZE    = 16
VAE_LEARNING_RATE = 1e-3

# ──────────────────────────────────────────────────────────────────────────────
#  FEATURE SELECTION
# ──────────────────────────────────────────────────────────────────────────────
IG_TOP_K = 30

# ──────────────────────────────────────────────────────────────────────────────
#  1D-CNN
# ──────────────────────────────────────────────────────────────────────────────
CNN_FILTERS_1     = 64
CNN_FILTERS_2     = 128
CNN_KERNEL_SIZE   = 3
CNN_POOL_SIZE     = 2
CNN_DENSE_UNITS   = 128
CNN_DROPOUT       = 0.0
CNN_LEARNING_RATE = 0.0001
CNN_BATCH_SIZE    = 64
CNN_EPOCHS        = 50
CNN_NUM_CLASSES   = 2

# ──────────────────────────────────────────────────────────────────────────────
#  CROSS-VALIDATION
# ──────────────────────────────────────────────────────────────────────────────
N_FOLDS     = 5
RANDOM_SEED = 42

# ──────────────────────────────────────────────────────────────────────────────
#  LABEL / METADATA  (UTSW_Glioma_Metadata TSV, replaces the old labels PDF)
# ──────────────────────────────────────────────────────────────────────────────
COL_SUBJECT_ID = "Subject ID"
COL_LABEL      = "1p19Q CODEL"
PATIENT_ID_COL = "patient_id"
UNKNOWN_LABEL  = -1

LABEL_POSITIVE_VALUES = {
    "co-deleted", "codeleted", "codel", "1p/19q codeleted", "yes", "positive", "1",
}
LABEL_NEGATIVE_VALUES = {
    "non co-deleted", "non-codeleted", "not codeleted", "intact", "no", "negative", "0",
}

# ──────────────────────────────────────────────────────────────────────────────
#  PATCH-BASED
# ──────────────────────────────────────────────────────────────────────────────
PATCH_SIZE               = (16, 16, 16)
PATCH_MIN_TUMOR_FRACTION = 0.50
PATCH_STRIDE             = None
PATCH_FEATURES_DIR       = os.path.join(OUTPUT_ROOT, 'patch_features')
PATCH_RESULTS_DIR        = os.path.join(OUTPUT_ROOT, 'patch_results')
PATCH_FEATURE_CLASSES    = ['firstorder', 'glcm', 'glszm', 'glrlm', 'ngtdm', 'gldm']

# ──────────────────────────────────────────────────────────────────────────────
#  CROSS-MODAL DIFFERENTIAL FEATURES (CMDF)
# ──────────────────────────────────────────────────────────────────────────────
CMDF_FEATURES_DIR  = os.path.join(OUTPUT_ROOT, 'cmdf_features')
CMDF_RESULTS_DIR   = os.path.join(OUTPUT_ROOT, 'cmdf_results')

# Which differential metrics to compute between modalities
# Options: 'variance', 'l2', 'cosine', 'outlier_deviation'
CMDF_METRICS = ['variance', 'l2', 'cosine', 'outlier_deviation']

# ──────────────────────────────────────────────────────────────────────────────
#  ATTENTION MIL MODEL
# ──────────────────────────────────────────────────────────────────────────────
MIL_HIDDEN_DIM     = 128
MIL_ATTENTION_DIM  = 64
MIL_DROPOUT        = 0.3
MIL_LEARNING_RATE  = 1e-4
MIL_BATCH_SIZE     = 1
MIL_EPOCHS         = 80


Overwriting config.py


## 3. Core Shared Modules

In [ ]:
%%writefile vae_augmentation.py
"""
vae_augmentation.py – VAE-based minority-class oversampling (unchanged).
"""
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from config import (VAE_LATENT_DIM, VAE_HIDDEN_UNITS, VAE_EPOCHS,
                    VAE_BATCH_SIZE, VAE_LEARNING_RATE, RANDOM_SEED)


class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim   = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim), seed=RANDOM_SEED)
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


def build_vae(input_dim, latent_dim=VAE_LATENT_DIM, hidden_units=None):
    if hidden_units is None:
        hidden_units = list(VAE_HIDDEN_UNITS)
    enc_in = keras.Input(shape=(input_dim,))
    x = enc_in
    for u in hidden_units:
        x = layers.Dense(u, activation='relu')(x)
    z_mean    = layers.Dense(latent_dim, name='z_mean')(x)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
    z         = Sampling()([z_mean, z_log_var])
    encoder   = Model(enc_in, [z_mean, z_log_var, z], name='encoder')
    dec_in = keras.Input(shape=(latent_dim,))
    x = dec_in
    for u in reversed(hidden_units):
        x = layers.Dense(u, activation='relu')(x)
    dec_out  = layers.Dense(input_dim, activation='linear')(x)
    decoder  = Model(dec_in, dec_out, name='decoder')

    class VAE(Model):
        def __init__(self, enc, dec, **kw):
            super().__init__(**kw)
            self.encoder = enc
            self.decoder = dec
            self.tl = keras.metrics.Mean(name='total_loss')
            self.rl = keras.metrics.Mean(name='recon_loss')
            self.kl = keras.metrics.Mean(name='kl_loss')
        @property
        def metrics(self):
            return [self.tl, self.rl, self.kl]
        def train_step(self, data):
            with tf.GradientTape() as tape:
                zm, zlv, z = self.encoder(data)
                rec = self.decoder(z)
                rl  = tf.reduce_mean(tf.reduce_sum(tf.reduce_mean(tf.square(data - rec), axis=-1)))
                kl  = -0.5 * tf.reduce_mean(tf.reduce_sum(1 + zlv - tf.square(zm) - tf.exp(zlv), axis=1))
                tot = rl + kl
            grads = tape.gradient(tot, self.trainable_weights)
            self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
            self.tl.update_state(tot); self.rl.update_state(rl); self.kl.update_state(kl)
            return {'total_loss': self.tl.result(), 'recon_loss': self.rl.result(), 'kl_loss': self.kl.result()}

    vae = VAE(encoder, decoder)
    vae.compile(optimizer=keras.optimizers.Adam(learning_rate=VAE_LEARNING_RATE))
    return vae, encoder, decoder


def augment_features(X, y, balance=True, epochs=VAE_EPOCHS,
                     batch_size=VAE_BATCH_SIZE, verbose=0):
    np.random.seed(RANDOM_SEED)
    tf.random.set_seed(RANDOM_SEED)
    classes, counts = np.unique(y, return_counts=True)
    minority_class  = classes[np.argmin(counts)]
    n_majority = counts.max()
    n_minority = counts.min()
    n_to_gen   = n_majority - n_minority
    if n_to_gen <= 0:
        print('  Classes already balanced.')
        return X, y
    X_min = X[y == minority_class]
    print(f'  VAE: training on {len(X_min)} minority samples, generating {n_to_gen}...')
    vae, _, decoder = build_vae(input_dim=X.shape[1])
    vae.fit(X_min, epochs=epochs, batch_size=batch_size, verbose=verbose)
    z = np.random.normal(0, 1, size=(n_to_gen, VAE_LATENT_DIM))
    X_syn = decoder.predict(z, verbose=0)
    y_syn = np.full(n_to_gen, minority_class)
    X_aug = np.vstack([X, X_syn])
    y_aug = np.concatenate([y, y_syn])
    print(f'  Augmented: {len(X)} -> {len(X_aug)} samples')
    return X_aug, y_aug

Overwriting vae_augmentation.py


In [ ]:
%%writefile feature_selection.py
"""
feature_selection.py – Information Gain feature selection (unchanged).
"""
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from config import IG_TOP_K, RANDOM_SEED


def compute_information_gain(X, y, feature_names=None, random_state=RANDOM_SEED):
    mi = mutual_info_classif(X, y, discrete_features=False, random_state=random_state)
    if feature_names is None:
        feature_names = [f'f{i}' for i in range(X.shape[1])]
    return pd.DataFrame({'feature': feature_names, 'ig_score': mi}
                        ).sort_values('ig_score', ascending=False).reset_index(drop=True)


def select_features(X, y, feature_names, top_k=IG_TOP_K):
    ig_df = compute_information_gain(X, y, feature_names)
    selected = ig_df['feature'].iloc[:top_k].tolist()
    idx = [feature_names.index(n) for n in selected]
    return X[:, idx], selected, ig_df

Overwriting feature_selection.py


In [ ]:
%%writefile model_cnn.py
"""
model_cnn.py – 1D Enhanced CNN (unchanged).
"""
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from config import (CNN_FILTERS_1, CNN_FILTERS_2, CNN_KERNEL_SIZE, CNN_POOL_SIZE,
                    CNN_DENSE_UNITS, CNN_DROPOUT, CNN_LEARNING_RATE,
                    CNN_BATCH_SIZE, CNN_EPOCHS, CNN_NUM_CLASSES, RANDOM_SEED)


def build_cnn(n_features):
    tf.random.set_seed(RANDOM_SEED)
    inp = keras.Input(shape=(n_features, 1))
    x   = layers.Conv1D(CNN_FILTERS_1, CNN_KERNEL_SIZE, activation='relu', padding='same')(inp)
    x   = layers.MaxPooling1D(CNN_POOL_SIZE)(x)
    x   = layers.Conv1D(CNN_FILTERS_2, CNN_KERNEL_SIZE, activation='relu', padding='same')(x)
    x   = layers.MaxPooling1D(CNN_POOL_SIZE)(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(CNN_DENSE_UNITS, activation='relu')(x)
    if CNN_DROPOUT > 0:
        x = layers.Dropout(CNN_DROPOUT)(x)
    out = layers.Dense(CNN_NUM_CLASSES, activation='softmax')(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(CNN_LEARNING_RATE),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


def train_cnn(model, X_train, y_train, X_val=None, y_val=None, verbose=0):
    X_tr = X_train[..., np.newaxis]
    val_data = (X_val[..., np.newaxis], y_val) if X_val is not None else None
    model.fit(X_tr, y_train, epochs=CNN_EPOCHS, batch_size=CNN_BATCH_SIZE,
              validation_data=val_data, verbose=verbose)


def predict_cnn(model, X):
    probs = model.predict(X[..., np.newaxis], verbose=0)
    return probs.argmax(axis=1), probs[:, 1]

Overwriting model_cnn.py


In [ ]:
%%writefile model_baselines.py
"""
model_baselines.py – KNN, DT, GNB, RF baselines (unchanged).
"""
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from config import RANDOM_SEED


def get_baseline_models():
    return {
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'DT' : DecisionTreeClassifier(random_state=RANDOM_SEED),
        'GNB': GaussianNB(),
        'RF' : RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
    }


def train_predict_baseline(clf, X_train, y_train, X_test):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    if hasattr(clf, 'predict_proba'):
        y_prob = clf.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred.astype(float)
    return y_pred, y_prob

Overwriting model_baselines.py


In [ ]:
%%writefile model_mil.py
"""
model_mil.py – Gated Attention MIL (unchanged).
"""
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from config import (MIL_HIDDEN_DIM, MIL_ATTENTION_DIM, MIL_DROPOUT,
                    MIL_LEARNING_RATE, MIL_EPOCHS, RANDOM_SEED)


class GatedAttention(layers.Layer):
    def __init__(self, hidden_dim, attention_dim, **kw):
        super().__init__(**kw)
        self.V = layers.Dense(attention_dim, activation='tanh')
        self.U = layers.Dense(attention_dim, activation='sigmoid')
        self.W = layers.Dense(1)
    def call(self, h):
        a = self.W(self.V(h) * self.U(h))
        a = tf.nn.softmax(a, axis=1)
        return tf.reduce_sum(a * h, axis=1), a


def build_mil_model(n_features, hidden_dim=MIL_HIDDEN_DIM,
                    attention_dim=MIL_ATTENTION_DIM, dropout=MIL_DROPOUT):
    tf.random.set_seed(RANDOM_SEED)
    inp = layers.Input(shape=(None, n_features))
    h = layers.Dense(hidden_dim, activation='relu')(inp)
    h = layers.Dropout(dropout)(h)
    h = layers.Dense(hidden_dim // 2, activation='relu')(h)
    h = layers.Dropout(dropout)(h)
    z, _ = GatedAttention(hidden_dim // 2, attention_dim)(h)
    out = layers.Dense(1, activation='sigmoid')(z)
    model = keras.Model(inputs=inp, outputs=out, name='AttentionMIL')
    model.compile(optimizer=keras.optimizers.Adam(MIL_LEARNING_RATE),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model


def train_mil(model, bags_train, labels_train, bags_val=None,
              labels_val=None, epochs=MIL_EPOCHS, verbose=0):
    classes, counts = np.unique(labels_train, return_counts=True)
    total = counts.sum()
    weight_map = {int(c): total / (len(classes) * n) for c, n in zip(classes, counts)}
    for epoch in range(epochs):
        for i in np.random.permutation(len(bags_train)):
            bag = bags_train[i][np.newaxis, ...]
            lbl = np.array([[labels_train[i]]], dtype=np.float32)
            sw  = np.array([[weight_map[int(labels_train[i])]]])
            model.train_on_batch(bag, lbl, sample_weight=sw)


def predict_mil(model, bags):
    probs = np.array([model.predict(b[np.newaxis, ...], verbose=0)[0, 0] for b in bags])
    return (probs >= 0.5).astype(int), probs

Overwriting model_mil.py


In [ ]:
%%writefile mil_aggregation.py
"""
mil_aggregation.py – Patch aggregation (unchanged).
"""
import numpy as np
import pandas as pd


def aggregate_patient_features(patch_df, aggregation_stats=None):
    if aggregation_stats is None:
        aggregation_stats = ['mean', 'std', 'min', 'max', 'median']
    meta_cols = ['patient_id', 'patch_idx', 'label']
    feat_cols = [c for c in patch_df.columns if c not in meta_cols]
    rows = []
    for pid, group in patch_df.groupby('patient_id'):
        row = {'patient_id': pid, 'label': int(group['label'].iloc[0]),
               'n_patches': len(group)}
        for stat in aggregation_stats:
            vals = getattr(group[feat_cols], stat)()
            if stat == 'std': vals = vals.fillna(0.0)
            for col in feat_cols:
                row[f'{col}_{stat}'] = vals[col]
        rows.append(row)
    agg_df = pd.DataFrame(rows)
    agg_meta = ['patient_id', 'label', 'n_patches']
    agg_feat = [c for c in agg_df.columns if c not in agg_meta]
    agg_df = agg_df[agg_meta + agg_feat]
    num = agg_df.select_dtypes(include=[np.number]).columns
    agg_df[num] = agg_df[num].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    print(f'  Aggregated {len(patch_df)} patches -> {len(agg_df)} patients, {len(agg_feat)} features')
    return agg_df


def prepare_mil_bags(patch_df):
    meta_cols = ['patient_id', 'patch_idx', 'label']
    feat_cols = [c for c in patch_df.columns if c not in meta_cols]
    bags, labels, patient_ids = [], [], []
    for pid, group in patch_df.groupby('patient_id'):
        bags.append(group[feat_cols].values.astype(np.float32))
        labels.append(int(group['label'].iloc[0]))
        patient_ids.append(pid)
    return bags, np.array(labels), patient_ids

Overwriting mil_aggregation.py


In [ ]:
%%writefile evaluate.py
"""
evaluate.py – Metrics, confusion matrices, ROC curves (unchanged).
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)
from config import RESULTS_DIR


def compute_metrics(y_true, y_pred, y_prob):
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float('nan')
    return {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
        'auc'      : auc,
    }


def aggregate_fold_metrics(fold_metrics):
    keys = fold_metrics[0].keys()
    agg  = {}
    for k in keys:
        vals = [m[k] for m in fold_metrics if not np.isnan(m[k])]
        agg[f'{k}_mean'] = float(np.mean(vals)) if vals else float('nan')
        agg[f'{k}_std']  = float(np.std(vals))  if vals else float('nan')
    return agg


def plot_confusion_matrix(y_true, y_pred, model_name, save_dir,
                          class_names=None):
    if class_names is None:
        class_names = ['Non-codeletion (0)', 'Codeletion (1)']
    os.makedirs(save_dir, exist_ok=True)
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {model_name}')
    plt.tight_layout()
    path = os.path.join(save_dir, f'cm_{model_name}.png')
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_roc_curves(roc_data, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    for name, (y_true, y_prob) in roc_data.items():
        if len(np.unique(y_true)) < 2: continue
        try:
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            auc = roc_auc_score(y_true, y_prob)
            ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
        except Exception: pass
    ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Random')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curves'); ax.legend(loc='lower right')
    plt.tight_layout()
    path = os.path.join(save_dir, 'roc_curves.png')
    fig.savefig(path, dpi=150)
    plt.close(fig)


def save_results_table(all_results, save_dir, prefix=''):
    os.makedirs(save_dir, exist_ok=True)
    rows = []
    for name, metrics in all_results.items():
        row = {'model': name}
        row.update(metrics)
        rows.append(row)
    df = pd.DataFrame(rows)
    fname = f'{prefix}metrics.csv' if prefix else 'metrics.csv'
    path  = os.path.join(save_dir, fname)
    df.to_csv(path, index=False)
    print(f'\n{"="*80}')
    print(f'  MODEL COMPARISON — 5-Fold CV  [{prefix.rstrip("_")}]')
    print(f'{"="*80}')
    for _, r in df.iterrows():
        print(f"\n  {r['model']}:")
        for m in ['accuracy','precision','recall','f1','auc']:
            mk = f'{m}_mean'; sk = f'{m}_std'
            if mk in r: print(f'    {m:>10s}: {r[mk]:.4f} ± {r[sk]:.4f}')
    print(f'{"="*80}')
    return df

Overwriting evaluate.py


## 3b. Demonstration and Testing of the Modules Completed

Before running the full 5-fold pipelines (which require the real MRI-derived feature CSVs), each core module built above is smoke-tested here on small **synthetic data**. This confirms every module imports cleanly, runs end-to-end, and produces sane outputs — independent of whether the real dataset is mounted yet.

**Modules demonstrated:** `feature_selection.py` · `vae_augmentation.py` · `model_baselines.py` · `model_cnn.py` · `mil_aggregation.py` · `model_mil.py` · `evaluate.py`

### 3b.1 Synthetic Demo Dataset

In [ ]:
import numpy as np

np.random.seed(42)

n_samples, n_features = 60, 40
X_demo = np.random.randn(n_samples, n_features).astype(np.float32)
y_demo = np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3])
X_demo[y_demo == 1] += 1.5  # inject class separation so models have signal to learn
feature_names_demo = [f'feat_{i}' for i in range(n_features)]

print(f'Synthetic demo dataset -> X: {X_demo.shape}, y: {y_demo.shape}')
print(f'Class balance: {dict(zip(*np.unique(y_demo, return_counts=True)))}')

Synthetic demo dataset -> X: (60, 40), y: (60,)
Class balance: {np.int64(0): np.int64(43), np.int64(1): np.int64(17)}


### 3b.2 Feature Selection — `feature_selection.py`

In [ ]:
from feature_selection import compute_information_gain, select_features

ig_df_demo = compute_information_gain(X_demo, y_demo, feature_names_demo)
print('Top 10 features by Information Gain:')
display(ig_df_demo.head(10))

X_sel_demo, selected_feats_demo, _ = select_features(X_demo, y_demo, feature_names_demo, top_k=15)
print(f'\nSelected {len(selected_feats_demo)} / {n_features} features -> X_sel shape: {X_sel_demo.shape}')

Top 10 features by Information Gain:


,feature,ig_score
0,feat_28,0.371757
1,feat_24,0.318274
2,feat_10,0.296209
3,feat_2,0.296209
4,feat_20,0.280193
5,feat_13,0.264006
6,feat_34,0.259286
7,feat_39,0.250272
8,feat_21,0.246749
9,feat_30,0.241057



Selected 15 / 40 features -> X_sel shape: (60, 15)


### 3b.3 VAE Minority-Class Augmentation — `vae_augmentation.py`

In [ ]:
from vae_augmentation import augment_features

before_counts = dict(zip(*np.unique(y_demo, return_counts=True)))
X_aug_demo, y_aug_demo = augment_features(X_sel_demo, y_demo, epochs=5, batch_size=8, verbose=0)
after_counts = dict(zip(*np.unique(y_aug_demo, return_counts=True)))

print(f'Before augmentation: {X_sel_demo.shape[0]} samples, class counts {before_counts}')
print(f'After  augmentation: {X_aug_demo.shape[0]} samples, class counts {after_counts}')

  VAE: training on 17 minority samples, generating 26...
  Augmented: 60 -> 86 samples
Before augmentation: 60 samples, class counts {np.int64(0): np.int64(43), np.int64(1): np.int64(17)}
After  augmentation: 86 samples, class counts {np.int64(0): np.int64(43), np.int64(1): np.int64(43)}


### 3b.4 Baseline Classifiers — `model_baselines.py`

In [ ]:
from sklearn.model_selection import train_test_split
from model_baselines import get_baseline_models, train_predict_baseline
from evaluate import compute_metrics

X_tr_demo, X_te_demo, y_tr_demo, y_te_demo = train_test_split(
    X_aug_demo, y_aug_demo, test_size=0.3, stratify=y_aug_demo, random_state=42)

baseline_demo_results = {}
roc_data_demo = {}
for name, clf in get_baseline_models().items():
    y_pred, y_prob = train_predict_baseline(clf, X_tr_demo, y_tr_demo, X_te_demo)
    m = compute_metrics(y_te_demo, y_pred, y_prob)
    baseline_demo_results[name] = m
    roc_data_demo[name] = (y_te_demo, y_prob)
    print(f'{name:>4s} -> acc={m["accuracy"]:.3f}  f1={m["f1"]:.3f}  auc={m["auc"]:.3f}')

 KNN -> acc=0.846  f1=0.867  auc=1.000
  DT -> acc=0.808  f1=0.839  auc=0.808
 GNB -> acc=1.000  f1=1.000  auc=1.000
  RF -> acc=1.000  f1=1.000  auc=1.000


### 3b.5 1D Enhanced CNN — `model_cnn.py`

In [ ]:
from model_cnn import build_cnn, train_cnn, predict_cnn

cnn_demo = build_cnn(n_features=X_tr_demo.shape[1])
train_cnn(cnn_demo, X_tr_demo, y_tr_demo, X_val=X_te_demo, y_val=y_te_demo, verbose=0)
y_pred_cnn_demo, y_prob_cnn_demo = predict_cnn(cnn_demo, X_te_demo)

m_cnn_demo = compute_metrics(y_te_demo, y_pred_cnn_demo, y_prob_cnn_demo)
baseline_demo_results['ECNN'] = m_cnn_demo
roc_data_demo['ECNN'] = (y_te_demo, y_prob_cnn_demo)
print(f'ECNN -> acc={m_cnn_demo["accuracy"]:.3f}  f1={m_cnn_demo["f1"]:.3f}  auc={m_cnn_demo["auc"]:.3f}')

ECNN -> acc=1.000  f1=1.000  auc=1.000


### 3b.6 Patch Aggregation & MIL Bag Preparation — `mil_aggregation.py`

In [ ]:
import pandas as pd
from mil_aggregation import aggregate_patient_features, prepare_mil_bags

n_patients, patches_per_patient, n_patch_feats = 10, 8, 12
rows = []
for pid in range(n_patients):
    label = int(pid % 2 == 0)
    for pidx in range(patches_per_patient):
        row = {'patient_id': f'P{pid:03d}', 'patch_idx': pidx, 'label': label}
        feats = np.random.randn(n_patch_feats) + (1.0 if label else 0.0)
        row.update({f'pf_{i}': feats[i] for i in range(n_patch_feats)})
        rows.append(row)
patch_df_demo = pd.DataFrame(rows)
print(f'Synthetic patch-level dataframe: {patch_df_demo.shape}')

agg_df_demo = aggregate_patient_features(patch_df_demo)
display(agg_df_demo.head())

bags_demo, labels_demo, pids_demo = prepare_mil_bags(patch_df_demo)
print(f'Prepared {len(bags_demo)} MIL bags, each bag shaped like {bags_demo[0].shape}')

Synthetic patch-level dataframe: (80, 15)
  Aggregated 80 patches -> 10 patients, 60 features


,patient_id,label,n_patches,pf_0_mean,pf_1_mean,pf_2_mean,pf_3_mean,pf_4_mean,pf_5_mean,pf_6_mean,...,pf_2_median,pf_3_median,pf_4_median,pf_5_median,pf_6_median,pf_7_median,pf_8_median,pf_9_median,pf_10_median,pf_11_median
0,P000,1,8,1.422638,1.384718,0.826468,1.644670,1.246282,0.919267,1.436399,...,0.688310,1.640821,1.424204,0.369037,1.358449,0.713360,1.099960,1.449304,1.006161,1.379811
1,P001,0,8,0.218161,0.191130,-0.030433,0.371234,0.160827,-0.573730,-0.222674,...,-0.011260,0.479163,0.151284,-0.582935,-0.096121,-0.164406,0.582779,-0.145948,0.086083,0.462319
2,P002,1,8,1.332955,0.570797,1.442232,1.016522,1.310186,0.329286,1.330534,...,1.570390,1.228111,1.553774,0.556500,1.039409,0.962135,1.186835,1.199225,1.372350,1.198162
3,P003,0,8,0.260019,0.814296,-0.042210,-0.431942,-0.048709,0.281284,-0.529666,...,0.197287,-0.493866,0.042282,0.244948,-0.868226,-0.486790,-0.180851,0.115211,0.113837,0.482609
4,P004,1,8,0.829345,1.059844,0.943449,0.884397,0.959483,1.388532,1.133575,...,0.856907,0.170174,0.952824,1.364420,1.057956,1.202340,1.396056,0.696659,1.299136,1.414845


Prepared 10 MIL bags, each bag shaped like (8, 12)


### 3b.7 Gated-Attention MIL — `model_mil.py`

In [ ]:
from model_mil import build_mil_model, train_mil, predict_mil

mil_demo = build_mil_model(n_features=n_patch_feats)
train_mil(mil_demo, bags_demo, labels_demo, epochs=5, verbose=0)
mil_pred_demo, mil_prob_demo = predict_mil(mil_demo, bags_demo)

m_mil_demo = compute_metrics(labels_demo, mil_pred_demo, mil_prob_demo)
print(f'Attention-MIL -> acc={m_mil_demo["accuracy"]:.3f}  f1={m_mil_demo["f1"]:.3f}  auc={m_mil_demo["auc"]:.3f}')

Attention-MIL -> acc=0.500  f1=0.667  auc=1.000


### 3b.8 Evaluation Utilities — `evaluate.py`

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from evaluate import aggregate_fold_metrics, plot_confusion_matrix, plot_roc_curves, save_results_table

DEMO_DIR = 'demo_test_output'
os.makedirs(DEMO_DIR, exist_ok=True)

# aggregate_fold_metrics expects a list of per-fold metric dicts — reuse two demo models as mock folds
mock_fold_metrics = [baseline_demo_results['RF'], baseline_demo_results['KNN']]
agg_demo = aggregate_fold_metrics(mock_fold_metrics)
print('aggregate_fold_metrics() -> mean ± std across mock folds:')
for k, v in agg_demo.items():
    print(f'  {k}: {v:.4f}')

plot_confusion_matrix(y_te_demo, y_pred_cnn_demo, 'ECNN_demo', DEMO_DIR)
plot_roc_curves(roc_data_demo, DEMO_DIR)
results_df_demo = save_results_table(baseline_demo_results, DEMO_DIR, prefix='demo_')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(mpimg.imread(os.path.join(DEMO_DIR, 'cm_ECNN_demo.png'))); axes[0].axis('off')
axes[0].set_title('Confusion Matrix (demo)')
axes[1].imshow(mpimg.imread(os.path.join(DEMO_DIR, 'roc_curves.png'))); axes[1].axis('off')
axes[1].set_title('ROC Curves (demo)')
plt.tight_layout()
plt.show()

aggregate_fold_metrics() -> mean ± std across mock folds:
  accuracy_mean: 0.9231
  accuracy_std: 0.0769
  precision_mean: 0.8824
  precision_std: 0.1176
  recall_mean: 1.0000
  recall_std: 0.0000
  f1_mean: 0.9333
  f1_std: 0.0667
  auc_mean: 1.0000
  auc_std: 0.0000

  MODEL COMPARISON — 5-Fold CV  [demo]

  KNN:

  DT:

  GNB:

  RF:

  ECNN:


### ✅ Module Demonstration Summary

| Module | Function(s) tested | Result |
|---|---|---|
| `feature_selection.py` | `compute_information_gain`, `select_features` | Ranked & selected top-k synthetic features |
| `vae_augmentation.py` | `augment_features` | Minority class oversampled to balance |
| `model_baselines.py` | `get_baseline_models`, `train_predict_baseline` | KNN / DT / GNB / RF trained & scored |
| `model_cnn.py` | `build_cnn`, `train_cnn`, `predict_cnn` | ECNN trained & scored on synthetic split |
| `mil_aggregation.py` | `aggregate_patient_features`, `prepare_mil_bags` | Patch-level rows aggregated to patient-level, bags built |
| `model_mil.py` | `build_mil_model`, `train_mil`, `predict_mil` | Attention-MIL trained & scored on synthetic bags |
| `evaluate.py` | `compute_metrics`, `aggregate_fold_metrics`, `plot_confusion_matrix`, `plot_roc_curves`, `save_results_table` | Metrics computed, confusion matrix & ROC plots generated |

All modules imported and executed without error on synthetic data — confirming they are ready to run against the real feature CSVs in the pipelines below.

## 4. Pipeline Runner Modules

In [ ]:
%%writefile pipeline_baseline.py
"""
pipeline_baseline.py – Whole-tumour baseline pipeline (5-fold CV).
"""
import os, sys
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from config import (FEATURES_DIR, RESULTS_DIR, N_FOLDS, RANDOM_SEED, IG_TOP_K)
from vae_augmentation import augment_features
from feature_selection import select_features
from model_cnn import build_cnn, train_cnn, predict_cnn
from model_baselines import get_baseline_models, train_predict_baseline
from evaluate import (compute_metrics, aggregate_fold_metrics,
                       plot_confusion_matrix, plot_roc_curves, save_results_table)


def run_baseline_pipeline():
    print('\n' + '='*70)
    print('  BASELINE PIPELINE (Whole-Tumour Radiomics)')
    print('='*70)

    csv_path = os.path.join(FEATURES_DIR, 'radiomics_features.csv')
    if not os.path.exists(csv_path):
        print(f'ERROR: {csv_path} not found. Run baseline feature extraction first.')
        return None

    df = pd.read_csv(csv_path)
    feature_cols = [c for c in df.columns if c not in ('patient_id', 'label')]
    X_all = np.nan_to_num(df[feature_cols].values.astype(np.float32))
    y_all = df['label'].values.astype(int)
    print(f'  Loaded: {len(df)} patients × {len(feature_cols)} features')
    print(f'  Classes: {dict(zip(*np.unique(y_all, return_counts=True)))}')

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    model_names = ['ECNN', 'KNN', 'DT', 'GNB', 'RF']
    all_fold_metrics = {n: [] for n in model_names}
    all_roc_data     = {n: ([], []) for n in model_names}

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X_all, y_all)):
        print(f'\n  --- FOLD {fold_idx+1}/{N_FOLDS} ---')
        X_tr, X_te = X_all[tr_idx], X_all[te_idx]
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]

        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_te_sc = scaler.transform(X_te)

        X_tr_aug, y_tr_aug = augment_features(X_tr_sc, y_tr, balance=True, verbose=0)
        X_tr_sel, sel_names, ig_df = select_features(X_tr_aug, y_tr_aug,
                                                      feature_cols, top_k=IG_TOP_K)
        sel_idx = [feature_cols.index(n) for n in sel_names]
        X_te_sel = X_te_sc[:, sel_idx]

        tf.random.set_seed(RANDOM_SEED + fold_idx)
        cnn = build_cnn(n_features=len(sel_names))
        train_cnn(cnn, X_tr_sel, y_tr_aug, X_te_sel, y_te, verbose=0)
        y_pred, y_prob = predict_cnn(cnn, X_te_sel)
        m = compute_metrics(y_te, y_pred, y_prob)
        all_fold_metrics['ECNN'].append(m)
        all_roc_data['ECNN'][0].extend(y_te)
        all_roc_data['ECNN'][1].extend(y_prob)
        print(f'    ECNN  Acc={m["accuracy"]:.3f} F1={m["f1"]:.3f} AUC={m["auc"]:.3f}')

        for name, clf in get_baseline_models().items():
            y_pred, y_prob = train_predict_baseline(clf, X_tr_sel, y_tr_aug, X_te_sel)
            m = compute_metrics(y_te, y_pred, y_prob)
            all_fold_metrics[name].append(m)
            all_roc_data[name][0].extend(y_te)
            all_roc_data[name][1].extend(y_prob)
            print(f'    {name:>4s}  Acc={m["accuracy"]:.3f} F1={m["f1"]:.3f} AUC={m["auc"]:.3f}')

    os.makedirs(RESULTS_DIR, exist_ok=True)
    agg = {n: aggregate_fold_metrics(all_fold_metrics[n]) for n in model_names}
    res_df = save_results_table(agg, RESULTS_DIR)

    roc_dict = {}
    for n in model_names:
        yt = np.array(all_roc_data[n][0])
        yp = np.array(all_roc_data[n][1])
        roc_dict[n] = (yt, yp)
        plot_confusion_matrix(yt, (yp >= 0.5).astype(int), n, RESULTS_DIR)
    plot_roc_curves(roc_dict, RESULTS_DIR)

    print('\n✅ Baseline pipeline complete!')
    return res_df

Overwriting pipeline_baseline.py


In [ ]:
%%writefile pipeline_patch.py
"""
pipeline_patch.py – Patch-only pipeline (5-fold CV, no CMDF).
"""
import os, sys
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from config import (PATCH_FEATURES_DIR, PATCH_RESULTS_DIR, N_FOLDS, RANDOM_SEED, IG_TOP_K)
from mil_aggregation import aggregate_patient_features, prepare_mil_bags
from feature_selection import select_features
from model_cnn import build_cnn, train_cnn, predict_cnn
from model_baselines import get_baseline_models, train_predict_baseline
from model_mil import build_mil_model, train_mil, predict_mil
from evaluate import (compute_metrics, aggregate_fold_metrics,
                       plot_confusion_matrix, plot_roc_curves, save_results_table)


def run_patch_pipeline():
    print('\n' + '='*70)
    print('  PATCH-BASED PIPELINE (Aggregated Stats + Attention MIL)')
    print('='*70)

    csv_path = os.path.join(PATCH_FEATURES_DIR, 'patch_radiomics_features.csv')
    if not os.path.exists(csv_path):
        print(f'ERROR: {csv_path} not found.')
        return None

    patch_df = pd.read_csv(csv_path)
    print(f'  Loaded {len(patch_df)} patches from {patch_df["patient_id"].nunique()} patients')

    agg_df = aggregate_patient_features(patch_df)
    meta_cols = ['patient_id', 'label', 'n_patches']
    feature_cols = [c for c in agg_df.columns if c not in meta_cols]
    X_all = np.nan_to_num(agg_df[feature_cols].values.astype(np.float32))
    y_all = agg_df['label'].values.astype(int)
    pid_all = agg_df['patient_id'].values

    bags_all, labels_bag, pids_bag = prepare_mil_bags(patch_df)
    pid_to_bag = {p: b for p, b in zip(pids_bag, bags_all)}

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    model_names = ['Patch_ECNN', 'Patch_KNN', 'Patch_DT', 'Patch_GNB', 'Patch_RF', 'Patch_MIL']
    all_fold_metrics = {n: [] for n in model_names}
    all_roc_data     = {n: ([], []) for n in model_names}

    os.makedirs(PATCH_RESULTS_DIR, exist_ok=True)

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X_all, y_all)):
        print(f'\n  --- FOLD {fold_idx+1}/{N_FOLDS} ---')
        X_tr, X_te = X_all[tr_idx], X_all[te_idx]
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]
        pids_tr = pid_all[tr_idx]
        pids_te = pid_all[te_idx]

        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_te_sc = scaler.transform(X_te)

        X_tr_sel, sel_names, _ = select_features(X_tr_sc, y_tr, feature_cols, top_k=IG_TOP_K)
        sel_idx = [feature_cols.index(n) for n in sel_names]
        X_te_sel = X_te_sc[:, sel_idx]

        tf.random.set_seed(RANDOM_SEED + fold_idx)
        cnn = build_cnn(n_features=len(sel_names))
        train_cnn(cnn, X_tr_sel, y_tr, X_te_sel, y_te, verbose=0)
        y_pred, y_prob = predict_cnn(cnn, X_te_sel)
        m = compute_metrics(y_te, y_pred, y_prob)
        all_fold_metrics['Patch_ECNN'].append(m)
        all_roc_data['Patch_ECNN'][0].extend(y_te)
        all_roc_data['Patch_ECNN'][1].extend(y_prob)
        print(f'    Patch_ECNN Acc={m["accuracy"]:.3f} F1={m["f1"]:.3f} AUC={m["auc"]:.3f}')

        for name, clf in get_baseline_models().items():
            pn = f'Patch_{name}'
            y_pred, y_prob = train_predict_baseline(clf, X_tr_sel, y_tr, X_te_sel)
            m = compute_metrics(y_te, y_pred, y_prob)
            all_fold_metrics[pn].append(m)
            all_roc_data[pn][0].extend(y_te)
            all_roc_data[pn][1].extend(y_prob)
            print(f'    {pn:>12s} Acc={m["accuracy"]:.3f} F1={m["f1"]:.3f} AUC={m["auc"]:.3f}')

        # MIL
        bags_tr = [pid_to_bag[p] for p in pids_tr if p in pid_to_bag]
        labs_tr = [y_all[i] for i, p in enumerate(pid_all) if i in tr_idx and p in pid_to_bag]
        bags_te = [pid_to_bag[p] for p in pids_te if p in pid_to_bag]
        labs_te = [y_all[i] for i, p in enumerate(pid_all) if i in te_idx and p in pid_to_bag]

        if bags_tr and bags_te:
            bag_scaler = StandardScaler().fit(np.vstack(bags_tr))
            bags_tr_sc = [bag_scaler.transform(b) for b in bags_tr]
            bags_te_sc = [bag_scaler.transform(b) for b in bags_te]
            tf.random.set_seed(RANDOM_SEED + fold_idx)
            mil = build_mil_model(n_features=bags_tr_sc[0].shape[1])
            train_mil(mil, bags_tr_sc, np.array(labs_tr), verbose=0)
            y_pred, y_prob = predict_mil(mil, bags_te_sc)
            m = compute_metrics(np.array(labs_te), y_pred, y_prob)
            all_fold_metrics['Patch_MIL'].append(m)
            all_roc_data['Patch_MIL'][0].extend(labs_te)
            all_roc_data['Patch_MIL'][1].extend(y_prob)
            print(f'    Patch_MIL  Acc={m["accuracy"]:.3f} F1={m["f1"]:.3f} AUC={m["auc"]:.3f}')

    agg = {n: aggregate_fold_metrics(v) for n, v in all_fold_metrics.items() if v}
    res_df = save_results_table(agg, PATCH_RESULTS_DIR)

    roc_dict = {}
    for n in model_names:
        yt = np.array(all_roc_data[n][0])
        yp = np.array(all_roc_data[n][1])
        if len(yt) > 0:
            roc_dict[n] = (yt, yp)
            plot_confusion_matrix(yt, (yp >= 0.5).astype(int), n, PATCH_RESULTS_DIR)
    plot_roc_curves(roc_dict, PATCH_RESULTS_DIR)

    print('\n✅ Patch pipeline complete!')
    return res_df

Overwriting pipeline_patch.py


In [ ]:
%%writefile pipeline_cmdf.py
"""
pipeline_cmdf.py – Patch + CMDF fused pipeline (5-fold CV).  ← NEW

Reads the fused_patch_cmdf_features.csv produced by run_feature_extraction_cmdf.py
and runs ECNN + KNN + DT + GNB + RF + Attention-MIL.

For MIL, the bag construction reuses the raw patch-level CSV so that attention
weights can still be computed per-patch; each patch's feature vector is
extended with the patient's CMDF summary before feeding into the attention module.
"""
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from config import (CMDF_FEATURES_DIR, CMDF_RESULTS_DIR,
                    PATCH_FEATURES_DIR, N_FOLDS, RANDOM_SEED, IG_TOP_K)
from mil_aggregation import prepare_mil_bags
from feature_selection import select_features
from model_cnn import build_cnn, train_cnn, predict_cnn
from model_baselines import get_baseline_models, train_predict_baseline
from model_mil import build_mil_model, train_mil, predict_mil
from evaluate import (compute_metrics, aggregate_fold_metrics,
                       plot_confusion_matrix, plot_roc_curves, save_results_table)


def _load_cmdf_features():
    fpath = os.path.join(CMDF_FEATURES_DIR, 'fused_patch_cmdf_features.csv')
    if not os.path.exists(fpath):
        raise FileNotFoundError(
            f'{fpath} not found. Run run_feature_extraction_cmdf.py first.')
    df = pd.read_csv(fpath)
    meta_cols   = ['patient_id', 'label', 'n_patches']
    feature_cols = [c for c in df.columns if c not in meta_cols]
    print(f'  Fused features: {len(feature_cols)} cols, {len(df)} patients')
    patch_feat = [c for c in feature_cols if not c.startswith('cmdf_')]
    cmdf_feat  = [c for c in feature_cols if c.startswith('cmdf_')]
    print(f'    Patch-agg: {len(patch_feat)}  |  CMDF: {len(cmdf_feat)}')
    return df, feature_cols


def _build_cmdf_augmented_bags(patch_df, fused_df):
    """
    Build MIL bags where each patch feature vector is extended with the
    patient's CMDF summary (patient-level broadcast onto each patch).
    This lets the attention mechanism attend to patches whose local
    radiomics most diverge cross-modally.
    """
    cmdf_cols = [c for c in fused_df.columns if c.startswith('cmdf_')]
    pid_to_cmdf = {
        row['patient_id']: fused_df.loc[fused_df['patient_id'] == row['patient_id'],
                                        cmdf_cols].values[0]
        for _, row in fused_df.iterrows()
    }

    meta_cols = ['patient_id', 'patch_idx', 'label']
    feat_cols = [c for c in patch_df.columns if c not in meta_cols]

    bags, labels, patient_ids = [], [], []
    for pid, group in patch_df.groupby('patient_id'):
        if pid not in pid_to_cmdf:
            continue
        patch_mat = group[feat_cols].values.astype(np.float32)  # (P, F_patch)
        cmdf_vec  = pid_to_cmdf[pid].astype(np.float32)          # (F_cmdf,)
        cmdf_rep  = np.tile(cmdf_vec, (len(patch_mat), 1))       # (P, F_cmdf)
        aug_mat   = np.hstack([patch_mat, cmdf_rep])              # (P, F_patch+F_cmdf)
        bags.append(aug_mat)
        labels.append(int(group['label'].iloc[0]))
        patient_ids.append(pid)

    return bags, np.array(labels), patient_ids


def run_cmdf_pipeline():
    print('\n' + '='*70)
    print('  PATCH + CMDF FUSED PIPELINE')
    print('='*70)

    fused_df, feature_cols = _load_cmdf_features()
    X_all  = np.nan_to_num(fused_df[feature_cols].values.astype(np.float32))
    y_all  = fused_df['label'].values.astype(int)
    pid_all = fused_df['patient_id'].values
    print(f'  Classes: {dict(zip(*np.unique(y_all, return_counts=True)))}')

    # Load raw patch CSV for CMDF-augmented MIL bags
    patch_csv = os.path.join(PATCH_FEATURES_DIR, 'patch_radiomics_features.csv')
    patch_df  = pd.read_csv(patch_csv) if os.path.exists(patch_csv) else None

    if patch_df is not None:
        aug_bags, aug_bag_labels, aug_bag_pids = \
            _build_cmdf_augmented_bags(patch_df, fused_df)
        pid_to_cmdf_bag = {p: b for p, b in zip(aug_bag_pids, aug_bags)}
    else:
        pid_to_cmdf_bag = {}
        print('  [WARN] patch_radiomics_features.csv not found — skipping CMDF_MIL')

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    model_names = ['CMDF_ECNN', 'CMDF_KNN', 'CMDF_DT', 'CMDF_GNB', 'CMDF_RF', 'CMDF_MIL']
    all_fold_metrics = {n: [] for n in model_names}
    all_roc_data     = {n: ([], []) for n in model_names}

    os.makedirs(CMDF_RESULTS_DIR, exist_ok=True)

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X_all, y_all)):
        print(f'\n  --- FOLD {fold_idx+1}/{N_FOLDS} ---')
        X_tr, X_te = X_all[tr_idx], X_all[te_idx]
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]
        pids_tr = pid_all[tr_idx]
        pids_te = pid_all[te_idx]

        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_te_sc = scaler.transform(X_te)

        X_tr_sel, sel_names, ig_df = select_features(
            X_tr_sc, y_tr, feature_cols, top_k=IG_TOP_K)
        sel_idx  = [feature_cols.index(n) for n in sel_names]
        X_te_sel = X_te_sc[:, sel_idx]

        # Report CMDF feature participation in top-K
        cmdf_in_topk = sum(1 for n in sel_names if n.startswith('cmdf_'))
        print(f'    Top-{IG_TOP_K} selected: {cmdf_in_topk} CMDF features '
              f'({cmdf_in_topk/IG_TOP_K*100:.0f}%)')

        # Save IG table for first fold
        if fold_idx == 0:
            ig_df.to_csv(os.path.join(CMDF_RESULTS_DIR, 'cmdf_ig_feature_ranking.csv'),
                         index=False)

        # ECNN
        tf.random.set_seed(RANDOM_SEED + fold_idx)
        cnn = build_cnn(n_features=len(sel_names))
        train_cnn(cnn, X_tr_sel, y_tr, X_te_sel, y_te, verbose=0)
        y_pred, y_prob = predict_cnn(cnn, X_te_sel)
        m = compute_metrics(y_te, y_pred, y_prob)
        all_fold_metrics['CMDF_ECNN'].append(m)
        all_roc_data['CMDF_ECNN'][0].extend(y_te)
        all_roc_data['CMDF_ECNN'][1].extend(y_prob)
        print(f'    CMDF_ECNN Acc={m["accuracy"]:.3f} F1={m["f1"]:.3f} AUC={m["auc"]:.3f}')

        # Baselines
        for name, clf in get_baseline_models().items():
            cn = f'CMDF_{name}'
            y_pred, y_prob = train_predict_baseline(clf, X_tr_sel, y_tr, X_te_sel)
            m = compute_metrics(y_te, y_pred, y_prob)
            all_fold_metrics[cn].append(m)
            all_roc_data[cn][0].extend(y_te)
            all_roc_data[cn][1].extend(y_prob)
            print(f'    {cn:>10s} Acc={m["accuracy"]:.3f} F1={m["f1"]:.3f} AUC={m["auc"]:.3f}')

        # CMDF-augmented Attention MIL
        bags_tr = [pid_to_cmdf_bag[p] for p in pids_tr if p in pid_to_cmdf_bag]
        labs_tr = [y_all[i] for i, p in enumerate(pid_all)
                   if i in tr_idx and p in pid_to_cmdf_bag]
        bags_te = [pid_to_cmdf_bag[p] for p in pids_te if p in pid_to_cmdf_bag]
        labs_te = [y_all[i] for i, p in enumerate(pid_all)
                   if i in te_idx and p in pid_to_cmdf_bag]

        if bags_tr and bags_te:
            bag_scaler = StandardScaler().fit(np.vstack(bags_tr))
            bags_tr_sc = [bag_scaler.transform(b) for b in bags_tr]
            bags_te_sc = [bag_scaler.transform(b) for b in bags_te]
            tf.random.set_seed(RANDOM_SEED + fold_idx)
            mil = build_mil_model(n_features=bags_tr_sc[0].shape[1])
            train_mil(mil, bags_tr_sc, np.array(labs_tr), verbose=0)
            y_pred, y_prob = predict_mil(mil, bags_te_sc)
            m = compute_metrics(np.array(labs_te), y_pred, y_prob)
            all_fold_metrics['CMDF_MIL'].append(m)
            all_roc_data['CMDF_MIL'][0].extend(labs_te)
            all_roc_data['CMDF_MIL'][1].extend(y_prob)
            print(f'    CMDF_MIL  Acc={m["accuracy"]:.3f} F1={m["f1"]:.3f} AUC={m["auc"]:.3f}')

    agg = {n: aggregate_fold_metrics(v) for n, v in all_fold_metrics.items() if v}
    res_df = save_results_table(agg, CMDF_RESULTS_DIR, prefix='cmdf_')

    roc_dict = {}
    for n in model_names:
        yt = np.array(all_roc_data[n][0])
        yp = np.array(all_roc_data[n][1])
        if len(yt) > 0:
            roc_dict[n] = (yt, yp)
            plot_confusion_matrix(yt, (yp >= 0.5).astype(int), n, CMDF_RESULTS_DIR)
    plot_roc_curves(roc_dict, CMDF_RESULTS_DIR)

    print('\n✅ Patch+CMDF pipeline complete!')
    return res_df

Overwriting pipeline_cmdf.py


## 5. Three-Way Comparison Module *(NEW)*

In [ ]:
%%writefile compare_all_pipelines.py
"""
compare_all_pipelines.py – Three-way comparison:
    Baseline  vs  Patch  vs  Patch+CMDF

Generates:
  • grouped bar charts  (Accuracy / F1 / AUC)
  • radar chart         (best model per pipeline)
  • gain table          (Patch and CMDF deltas over Baseline)
  • CMDF feature importance plot (which CMDF metrics contribute most)
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from config import RESULTS_DIR, PATCH_RESULTS_DIR, CMDF_RESULTS_DIR

COMPARISON_DIR = os.path.join(RESULTS_DIR, 'three_way_comparison')

PALETTE = {
    'Baseline' : '#4C72B0',
    'Patch'    : '#DD8452',
    'Patch+CMDF': '#55A868',
}


def _load(path, label_prefix=None):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    if label_prefix:
        df['pipeline'] = label_prefix
    return df


def _best_model(df):
    """Return the row with the highest mean F1."""
    return df.loc[df['f1_mean'].idxmax()]


def plot_grouped_bars(b_df, p_df, c_df):
    """Grouped bar chart comparing all models across pipelines."""
    metrics = ['accuracy', 'f1', 'auc']
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Rename patch/cmdf models to a canonical base name for alignment
    def _canon(name):
        for prefix in ('CMDF_', 'Patch_'):
            if name.startswith(prefix):
                return name[len(prefix):]
        return name

    base_models  = [_canon(m) for m in b_df['model']]
    patch_models = [_canon(m) for m in p_df['model']]
    cmdf_models  = [_canon(m) for m in c_df['model']]
    all_models   = list(dict.fromkeys(base_models + patch_models + cmdf_models))

    x     = np.arange(len(all_models))
    width = 0.26

    for ax, metric in zip(axes, metrics):
        mk = f'{metric}_mean'
        sk = f'{metric}_std'

        def _vals(df, canon_models):
            row_map = {_canon(r['model']): r for _, r in df.iterrows()}
            means = [row_map[m][mk] if m in row_map else 0 for m in all_models]
            stds  = [row_map[m][sk] if m in row_map else 0 for m in all_models]
            return means, stds

        bm, bs = _vals(b_df, base_models)
        pm, ps = _vals(p_df, patch_models)
        cm, cs = _vals(c_df, cmdf_models)

        ax.bar(x - width, bm, width, yerr=bs, capsize=3,
               label='Baseline', color=PALETTE['Baseline'], alpha=0.85)
        ax.bar(x,         pm, width, yerr=ps, capsize=3,
               label='Patch', color=PALETTE['Patch'], alpha=0.85)
        ax.bar(x + width, cm, width, yerr=cs, capsize=3,
               label='Patch+CMDF', color=PALETTE['Patch+CMDF'], alpha=0.85)

        ax.set_xlabel('Model'); ax.set_ylabel(metric.upper())
        ax.set_title(metric.upper()); ax.set_xticks(x)
        ax.set_xticklabels(all_models, rotation=40, ha='right')
        ax.set_ylim(0, 1.12); ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Three-Way Pipeline Comparison — 5-Fold CV', fontsize=14, y=1.01)
    plt.tight_layout()
    path = os.path.join(COMPARISON_DIR, 'three_way_bar_chart.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved grouped bar chart -> {path}')


def plot_radar(b_df, p_df, c_df):
    """Radar chart comparing best model from each pipeline."""
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    labels  = [m.upper() for m in metrics]

    rows = {
        'Baseline'  : _best_model(b_df),
        'Patch'     : _best_model(p_df),
        'Patch+CMDF': _best_model(c_df),
    }

    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for label, row in rows.items():
        vals = [row[f'{m}_mean'] for m in metrics] + [row[f'{metrics[0]}_mean']]
        ax.fill(angles, vals, alpha=0.18, color=PALETTE[label])
        ax.plot(angles, vals, 'o-', label=f'{label} ({row["model"]})',
                color=PALETTE[label], linewidth=2)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylim(0.5, 1.0)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=10)
    ax.set_title('Best-Model Radar Comparison', pad=20, fontsize=13)
    plt.tight_layout()
    path = os.path.join(COMPARISON_DIR, 'three_way_radar.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved radar chart -> {path}')


def _gain_table(b_df, p_df, c_df):
    """Build a delta table: how much each pipeline improves over Baseline."""
    metrics = ['accuracy', 'f1', 'auc']
    b_best  = _best_model(b_df)
    p_best  = _best_model(p_df)
    c_best  = _best_model(c_df)

    rows = []
    for label, row in [('Baseline', b_best), ('Patch', p_best), ('Patch+CMDF', c_best)]:
        r = {'Pipeline': label, 'Best Model': row['model']}
        for m in metrics:
            r[f'{m.upper()} (mean)'] = round(row[f'{m}_mean'], 4)
            if label != 'Baseline':
                r[f'Δ{m.upper()} vs Baseline'] = round(
                    row[f'{m}_mean'] - b_best[f'{m}_mean'], 4)
        rows.append(r)

    return pd.DataFrame(rows)


def plot_cmdf_feature_importance():
    """
    Bar chart of top CMDF features from the IG ranking saved during CMDF training.
    Shows which differential metric (variance/l2/cosine/outlier) contributes most.
    """
    ig_path = os.path.join(CMDF_RESULTS_DIR, 'cmdf_ig_feature_ranking.csv')
    if not os.path.exists(ig_path):
        print('  [SKIP] CMDF IG ranking not found.')
        return

    ig_df = pd.read_csv(ig_path)
    cmdf_only = ig_df[ig_df['feature'].str.startswith('cmdf_')].head(30)

    def _metric_type(name):
        for t in ('variance', 'l2', 'cosine', 'outlier'):
            if t in name:
                return t
        return 'other'

    cmdf_only = cmdf_only.copy()
    cmdf_only['metric_type'] = cmdf_only['feature'].apply(_metric_type)

    color_map = {'variance': '#4C72B0', 'l2': '#DD8452',
                 'cosine': '#55A868', 'outlier': '#C44E52', 'other': '#8172B3'}
    colors = cmdf_only['metric_type'].map(color_map)

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(cmdf_only['feature'], cmdf_only['ig_score'], color=colors)
    ax.invert_yaxis()
    ax.set_xlabel('Information Gain Score')
    ax.set_title('CMDF Feature Importance (Top-30 CMDF features by IG)')

    legend_patches = [mpatches.Patch(color=c, label=t.capitalize())
                      for t, c in color_map.items() if t != 'other']
    ax.legend(handles=legend_patches, loc='lower right')
    plt.tight_layout()
    path = os.path.join(COMPARISON_DIR, 'cmdf_feature_importance.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved CMDF feature importance -> {path}')


def run_three_way_comparison():
    os.makedirs(COMPARISON_DIR, exist_ok=True)

    b_df = _load(os.path.join(RESULTS_DIR,       'metrics.csv'),      'Baseline')
    p_df = _load(os.path.join(PATCH_RESULTS_DIR, 'metrics.csv'),      'Patch')
    c_df = _load(os.path.join(CMDF_RESULTS_DIR,  'cmdf_metrics.csv'), 'Patch+CMDF')

    missing = [name for name, df in [('Baseline', b_df), ('Patch', p_df),
                                      ('Patch+CMDF', c_df)] if df is None]
    if missing:
        print(f'[WARN] Missing results for: {missing}. Run those pipelines first.')

    available = {k: v for k, v in [('Baseline', b_df),
                                    ('Patch', p_df),
                                    ('Patch+CMDF', c_df)] if v is not None}

    if len(available) < 2:
        print('ERROR: Need at least 2 pipelines to compare.')
        return

    b = available.get('Baseline', next(iter(available.values())))
    p = available.get('Patch',     b)
    c = available.get('Patch+CMDF', p)

    print('\n  Generating three-way comparison charts...')
    plot_grouped_bars(b, p, c)
    plot_radar(b, p, c)
    plot_cmdf_feature_importance()

    # Save gain table
    gain_df = _gain_table(b, p, c)
    gain_path = os.path.join(COMPARISON_DIR, 'pipeline_gain_table.csv')
    gain_df.to_csv(gain_path, index=False)
    print(f'  Saved gain table -> {gain_path}')

    print('\n' + '='*70)
    print('  PIPELINE GAIN SUMMARY')
    print('='*70)
    print(gain_df.to_string(index=False))
    print('\n✅ Three-way comparison complete!')
    print(f'   All outputs -> {COMPARISON_DIR}')


if __name__ == '__main__':
    run_three_way_comparison()

Overwriting compare_all_pipelines.py


## 6. Run All Pipelines

Execute cells below in order. Each pipeline saves its own results directory, then the comparison block aggregates everything.

In [ ]:
# ── PIPELINE 1: Baseline (whole-tumour radiomics) ─────────────────────────
# Requires output/features/radiomics_features.csv from the baseline feature extraction notebook.
from pipeline_baseline import run_baseline_pipeline
baseline_results = run_baseline_pipeline()


  BASELINE PIPELINE (Whole-Tumour Radiomics)
  Loaded: 62 patients × 214 features
  Classes: {np.int64(0): np.int64(10), np.int64(1): np.int64(52)}

  --- FOLD 1/5 ---
  VAE: training on 8 minority samples, generating 33...
  Augmented: 49 -> 82 samples
    ECNN  Acc=0.769 F1=0.857 AUC=0.909
     KNN  Acc=0.769 F1=0.857 AUC=0.909
      DT  Acc=0.692 F1=0.800 AUC=0.614
     GNB  Acc=0.769 F1=0.842 AUC=0.909
      RF  Acc=0.769 F1=0.857 AUC=0.864

  --- FOLD 2/5 ---
  VAE: training on 8 minority samples, generating 33...


  Augmented: 49 -> 82 samples
    ECNN  Acc=0.692 F1=0.800 AUC=0.773
     KNN  Acc=0.615 F1=0.762 AUC=0.523
      DT  Acc=0.538 F1=0.700 AUC=0.318
     GNB  Acc=0.692 F1=0.800 AUC=0.523
      RF  Acc=0.615 F1=0.762 AUC=0.545

  --- FOLD 3/5 ---
  VAE: training on 8 minority samples, generating 34...
  Augmented: 50 -> 84 samples
    ECNN  Acc=0.917 F1=0.952 AUC=0.500
     KNN  Acc=0.833 F1=0.900 AUC=0.850
      DT  Acc=0.833 F1=0.900 AUC=0.700
     GNB  Acc=0.917 F1=0.952 AUC=0.725
      RF  Acc=1.000 F1=1.000 AUC=1.000

  --- FOLD 4/5 ---
  VAE: training on 8 minority samples, generating 34...
  Augmented: 50 -> 84 samples
    ECNN  Acc=0.917 F1=0.952 AUC=1.000
     KNN  Acc=0.917 F1=0.952 AUC=1.000
      DT  Acc=0.917 F1=0.952 AUC=0.750
     GNB  Acc=0.917 F1=0.952 AUC=1.000
      RF  Acc=0.917 F1=0.952 AUC=0.975

  --- FOLD 5/5 ---
  VAE: training on 8 minority samples, generating 34...
  Augmented: 50 -> 84 samples
    ECNN  Acc=0.833 F1=0.889 AUC=1.000
     KNN  Acc=0.917 F1=0.947

In [ ]:
# ── PIPELINE 2: Patch-only ────────────────────────────────────────────────
# Requires output/patch_features/patch_radiomics_features.csv
from pipeline_patch import run_patch_pipeline
patch_results = run_patch_pipeline()


  PATCH-BASED PIPELINE (Aggregated Stats + Attention MIL)
  Loaded 1645 patches from 62 patients
  Aggregated 1645 patches -> 62 patients, 930 features

  --- FOLD 1/5 ---
    Patch_ECNN Acc=0.846 F1=0.917 AUC=0.864
       Patch_KNN Acc=0.692 F1=0.800 AUC=0.614
        Patch_DT Acc=0.692 F1=0.800 AUC=0.614
       Patch_GNB Acc=0.615 F1=0.737 AUC=0.614
        Patch_RF Acc=0.769 F1=0.857 AUC=0.500
    Patch_MIL  Acc=0.769 F1=0.857 AUC=0.818

  --- FOLD 2/5 ---
    Patch_ECNN Acc=0.846 F1=0.917 AUC=0.273
       Patch_KNN Acc=0.769 F1=0.870 AUC=0.318
        Patch_DT Acc=0.692 F1=0.800 AUC=0.614
       Patch_GNB Acc=0.538 F1=0.667 AUC=0.455
        Patch_RF Acc=0.769 F1=0.870 AUC=0.318
    Patch_MIL  Acc=0.769 F1=0.857 AUC=0.500

  --- FOLD 3/5 ---
    Patch_ECNN Acc=0.833 F1=0.909 AUC=0.400
       Patch_KNN Acc=0.833 F1=0.909 AUC=0.775
        Patch_DT Acc=0.833 F1=0.909 AUC=0.500
       Patch_GNB Acc=0.333 F1=0.333 AUC=0.850
        Patch_RF Acc=0.833 F1=0.909 AUC=1.000
    Patch_MIL  

In [ ]:
# ── PIPELINE 3: Patch + CMDF (new) ────────────────────────────────────────
# Requires output/cmdf_features/fused_patch_cmdf_features.csv
# Run feature_extraction_cmdf.ipynb first if this file is missing.
from pipeline_cmdf import run_cmdf_pipeline
cmdf_results = run_cmdf_pipeline()


  PATCH + CMDF FUSED PIPELINE
  Fused features: 1212 cols, 62 patients
    Patch-agg: 930  |  CMDF: 282
  Classes: {np.int64(0): np.int64(10), np.int64(1): np.int64(52)}

  --- FOLD 1/5 ---
    Top-30 selected: 14 CMDF features (47%)
    CMDF_ECNN Acc=0.846 F1=0.917 AUC=0.909
      CMDF_KNN Acc=0.846 F1=0.909 AUC=0.886
       CMDF_DT Acc=0.923 F1=0.952 AUC=0.955
      CMDF_GNB Acc=0.615 F1=0.737 AUC=0.614
       CMDF_RF Acc=0.846 F1=0.909 AUC=0.909
    CMDF_MIL  Acc=0.769 F1=0.857 AUC=0.818

  --- FOLD 2/5 ---
    Top-30 selected: 24 CMDF features (80%)
    CMDF_ECNN Acc=0.846 F1=0.917 AUC=0.318
      CMDF_KNN Acc=0.615 F1=0.762 AUC=0.477
       CMDF_DT Acc=0.923 F1=0.957 AUC=0.750
      CMDF_GNB Acc=0.538 F1=0.667 AUC=0.545
       CMDF_RF Acc=0.846 F1=0.909 AUC=0.682
    CMDF_MIL  Acc=0.692 F1=0.800 AUC=0.364

  --- FOLD 3/5 ---
    Top-30 selected: 12 CMDF features (40%)
    CMDF_ECNN Acc=0.833 F1=0.909 AUC=0.900
      CMDF_KNN Acc=0.833 F1=0.909 AUC=0.950
       CMDF_DT Acc=0.833 F

In [ ]:
# ── THREE-WAY COMPARISON ──────────────────────────────────────────────────
from compare_all_pipelines import run_three_way_comparison
run_three_way_comparison()


  Generating three-way comparison charts...
  Saved grouped bar chart -> /content/drive/MyDrive/Capstone_paper_001/output/results/three_way_comparison/three_way_bar_chart.png
  Saved radar chart -> /content/drive/MyDrive/Capstone_paper_001/output/results/three_way_comparison/three_way_radar.png
  Saved CMDF feature importance -> /content/drive/MyDrive/Capstone_paper_001/output/results/three_way_comparison/cmdf_feature_importance.png
  Saved gain table -> /content/drive/MyDrive/Capstone_paper_001/output/results/three_way_comparison/pipeline_gain_table.csv

  PIPELINE GAIN SUMMARY
  Pipeline Best Model  ACCURACY (mean)  F1 (mean)  AUC (mean)  ΔACCURACY vs Baseline  ΔF1 vs Baseline  ΔAUC vs Baseline
  Baseline         RF           0.8269     0.8921      0.8768                    NaN              NaN               NaN
     Patch  Patch_MIL           0.8744     0.9238      0.8536                 0.0474           0.0317           -0.0232
Patch+CMDF    CMDF_RF           0.8885     0.9359    

## 7. Inline Results Viewer

Display all generated comparison plots directly in the notebook.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from config import RESULTS_DIR, PATCH_RESULTS_DIR, CMDF_RESULTS_DIR

COMPARISON_DIR = os.path.join(RESULTS_DIR, 'three_way_comparison')

def show_image(path, title=''):
    if not os.path.exists(path):
        print(f'[Not found] {path}')
        return
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# ── Main comparison plots ──
show_image(os.path.join(COMPARISON_DIR, 'three_way_bar_chart.png'),
           'Three-Way Grouped Bar Chart')
show_image(os.path.join(COMPARISON_DIR, 'three_way_radar.png'),
           'Best-Model Radar Chart')
show_image(os.path.join(COMPARISON_DIR, 'cmdf_feature_importance.png'),
           'CMDF Feature Importance (Information Gain)')

# ── Gain Table ──
gain_path = os.path.join(COMPARISON_DIR, 'pipeline_gain_table.csv')
if os.path.exists(gain_path):
    print('\n── Pipeline Gain Table ──')
    display(pd.read_csv(gain_path))

# ── Individual pipeline ROC curves ──
for label, d in [('Baseline', RESULTS_DIR),
                  ('Patch-Only', PATCH_RESULTS_DIR),
                  ('Patch+CMDF', CMDF_RESULTS_DIR)]:
    show_image(os.path.join(d, 'roc_curves.png'), f'ROC Curves — {label}')


── Pipeline Gain Table ──


,Pipeline,Best Model,ACCURACY (mean),F1 (mean),AUC (mean),ΔACCURACY vs Baseline,ΔF1 vs Baseline,ΔAUC vs Baseline
0,Baseline,RF,0.8269,0.8921,0.8768,NaN,NaN,NaN
1,Patch,Patch_MIL,0.8744,0.9238,0.8536,0.0474,0.0317,-0.0232
2,Patch+CMDF,CMDF_RF,0.8885,0.9359,0.8982,0.0615,0.0439,0.0214


## 8. CMDF Ablation Study

Quickly test which differential metrics matter most by individually disabling each one and observing the ECNN F1 drop.

In [ ]:
%%writefile ablation_cmdf.py
"""
ablation_cmdf.py – Ablation study: drop one CMDF metric at a time.
Reports ECNN F1 for each ablated configuration.
"""
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from config import (PATCH_FEATURES_DIR, CMDF_FEATURES_DIR, CMDF_RESULTS_DIR,
                    CMDF_METRICS, N_FOLDS, RANDOM_SEED, IG_TOP_K)
from mil_aggregation import aggregate_patient_features
from feature_selection import select_features
from model_cnn import build_cnn, train_cnn, predict_cnn
from evaluate import compute_metrics, aggregate_fold_metrics


def _run_single_config(patch_df, active_metrics, label):
    from cmdf_features import compute_cmdf_from_patch_df, fuse_patch_and_cmdf

    agg_df  = aggregate_patient_features(patch_df)
    cmdf_df = compute_cmdf_from_patch_df(patch_df, active_metrics=active_metrics)
    fused   = fuse_patch_and_cmdf(agg_df, cmdf_df)

    meta = ['patient_id', 'label', 'n_patches']
    fcols = [c for c in fused.columns if c not in meta]
    X = np.nan_to_num(fused[fcols].values.astype(np.float32))
    y = fused['label'].values.astype(int)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    fold_metrics = []
    for fi, (tri, tei) in enumerate(skf.split(X, y)):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tri])
        Xte = scaler.transform(X[tei])
        Xtr_s, sel, _ = select_features(Xtr, y[tri], fcols, top_k=IG_TOP_K)
        si = [fcols.index(n) for n in sel]
        Xte_s = Xte[:, si]
        tf.random.set_seed(RANDOM_SEED + fi)
        cnn = build_cnn(n_features=len(sel))
        train_cnn(cnn, Xtr_s, y[tri], Xte_s, y[tei], verbose=0)
        yp, ypr = predict_cnn(cnn, Xte_s)
        fold_metrics.append(compute_metrics(y[tei], yp, ypr))

    agg = aggregate_fold_metrics(fold_metrics)
    return {'Config': label,
            'Active Metrics': str(active_metrics),
            'F1':  round(agg['f1_mean'],  4),
            'AUC': round(agg['auc_mean'], 4),
            'Acc': round(agg['accuracy_mean'], 4)}


def run_ablation():
    patch_csv = os.path.join(PATCH_FEATURES_DIR, 'patch_radiomics_features.csv')
    if not os.path.exists(patch_csv):
        print('ERROR: patch_radiomics_features.csv not found.')
        return
    patch_df = pd.read_csv(patch_csv)

    all_metrics = list(CMDF_METRICS)
    configs     = [('All CMDF', all_metrics)]
    for m in all_metrics:
        configs.append((f'Drop {m}', [x for x in all_metrics if x != m]))
    configs.append(('No CMDF', []))   # patch-only baseline

    print('\nRunning CMDF ablation study...')
    results = []
    for label, active in configs:
        print(f'  Config: {label}')
        row = _run_single_config(patch_df, active if active else None, label)
        results.append(row)
        print(f'    F1={row["F1"]:.4f}  AUC={row["AUC"]:.4f}  Acc={row["Acc"]:.4f}')

    abl_df = pd.DataFrame(results)
    out = os.path.join(CMDF_RESULTS_DIR, 'ablation_cmdf.csv')
    os.makedirs(CMDF_RESULTS_DIR, exist_ok=True)
    abl_df.to_csv(out, index=False)
    print(f'\nAblation results saved -> {out}')
    print(abl_df.to_string(index=False))
    return abl_df


if __name__ == '__main__':
    run_ablation()

Overwriting ablation_cmdf.py


In [ ]:
# Run ablation (optional — takes ~5× longer than a single fold run)
# Uncomment to execute:
from ablation_cmdf import run_ablation
ablation_df = run_ablation()

print('Ablation study ready. Uncomment the lines above to run it.')
print('Results will be saved to output/cmdf_results/ablation_cmdf.csv')


Running CMDF ablation study...
  Config: All CMDF
  Aggregated 1645 patches -> 62 patients, 930 features
  CMDF: 93 shared feature dimensions found.
  Active differential metrics: ['variance', 'l2', 'cosine', 'outlier_deviation']
  CMDF shape: (62, 842)  (840 differential features per patient)
  Fused feature matrix: 62 patients, 1773 total columns (840 CMDF columns)
    F1=0.9121  AUC=0.7691  Acc=0.8385
  Config: Drop variance
  Aggregated 1645 patches -> 62 patients, 930 features
  CMDF: 93 shared feature dimensions found.
  Active differential metrics: ['l2', 'cosine', 'outlier_deviation']
  CMDF shape: (62, 563)  (561 differential features per patient)
  Fused feature matrix: 62 patients, 1494 total columns (561 CMDF columns)
    F1=0.9121  AUC=0.7773  Acc=0.8385
  Config: Drop l2
  Aggregated 1645 patches -> 62 patients, 930 features
  CMDF: 93 shared feature dimensions found.
  Active differential metrics: ['variance', 'cosine', 'outlier_deviation']
  CMDF shape: (62, 563)  (561

In [ ]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.6 MB/s eta 0:00:00


In [ ]:
"""
Cell 31 — Model Export
Fits a final scaler + IG selector + best classifier on ALL available data
(no fold splitting) and saves artifacts to brain_mri_model/.
Run AFTER all three pipelines have completed so that the CSV results exist.
"""
import os, json as _json, joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

from config import (CMDF_FEATURES_DIR, PATCH_FEATURES_DIR,
                    CMDF_RESULTS_DIR, PATCH_RESULTS_DIR, RESULTS_DIR,
                    N_FOLDS, RANDOM_SEED, IG_TOP_K, CMDF_METRICS)
from feature_selection import select_features
from model_baselines import get_baseline_models

# MODEL_DIR = "brain_mri_model"
MODEL_DIR = "/content/drive/MyDrive/Capstone_paper_001/brain_mri_model"
os.makedirs(MODEL_DIR, exist_ok=True)

# ── 1. LOAD FUSED FEATURES (same CSV the CMDF pipeline used) ────────────
fused_path = os.path.join(CMDF_FEATURES_DIR, "fused_patch_cmdf_features.csv")
if not os.path.exists(fused_path):
    raise FileNotFoundError(
        f"{fused_path} not found. Run feature_extraction_cmdf.ipynb first.")

df = pd.read_csv(fused_path)
meta_cols    = ["patient_id", "label", "n_patches"]
feature_cols = [c for c in df.columns if c not in meta_cols]
X_all        = np.nan_to_num(df[feature_cols].values.astype(np.float32))
y_all        = df["label"].values.astype(int)
print(f"Loaded: {len(df)} patients × {len(feature_cols)} features")
print(f"Classes: {dict(zip(*np.unique(y_all, return_counts=True)))}")

# ── 2. FIT SCALER ON ALL DATA ────────────────────────────────────────────
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_all)

# ── 3. IG FEATURE SELECTION ON ALL DATA ─────────────────────────────────
X_sel, sel_names, ig_df = select_features(
    X_sc, y_all, feature_cols, top_k=IG_TOP_K)
print(f"Selected {len(sel_names)} features (top-{IG_TOP_K} by IG)")

cmdf_count = sum(1 for n in sel_names if n.startswith("cmdf_"))
print(f"  → {cmdf_count} CMDF  |  {len(sel_names)-cmdf_count} patch-agg")

# ── 4. PICK BEST CLASSIFIER FROM CV RESULTS ─────────────────────────────
cmdf_metrics_csv = os.path.join(CMDF_RESULTS_DIR, "cmdf_metrics.csv")
if os.path.exists(cmdf_metrics_csv):
    res_df        = pd.read_csv(cmdf_metrics_csv)
    best_row      = res_df.loc[res_df["auc_mean"].idxmax()]
    best_clf_name = best_row["model"]
    print(f"\nBest model from CV: {best_clf_name} "
          f"(AUC={best_row['auc_mean']:.4f} ± {best_row['auc_std']:.4f})")
else:
    print("\n[INFO] No saved metrics found — running cross-validation now...")
    best_clf_name, best_auc = None, -1
    for name, clf in get_baseline_models().items():
        aucs = cross_val_score(clf, X_sel, y_all, cv=N_FOLDS,
                               scoring="roc_auc", n_jobs=-1)
        print(f"  {name:>4s}  AUC={aucs.mean():.4f} ± {aucs.std():.4f}")
        if aucs.mean() > best_auc:
            best_auc, best_clf_name = aucs.mean(), f"CMDF_{name}"
    print(f"\nBest model: {best_clf_name}  AUC={best_auc:.4f}")

# ── 5. REFIT BEST CLASSIFIER ON ALL DATA ────────────────────────────────
base_name = best_clf_name.replace("CMDF_", "").replace("Patch_", "")
all_clfs  = get_baseline_models()
if base_name in all_clfs:
    best_clf = all_clfs[base_name]
else:
    print(f"[WARN] Could not map {best_clf_name} to a baseline — defaulting to RF")
    best_clf      = all_clfs["RF"]
    best_clf_name = "CMDF_RF"

best_clf.fit(X_sel, y_all)
print(f"\nRefitted {best_clf_name} on all {len(y_all)} patients.")

# ── 6. SAVE ARTIFACTS ───────────────────────────────────────────────────
joblib.dump(scaler,    os.path.join(MODEL_DIR, "scaler.pkl"))
joblib.dump(sel_names, os.path.join(MODEL_DIR, "sel_names.pkl"))
joblib.dump(best_clf,  os.path.join(MODEL_DIR, "best_clf.pkl"))

meta = {
    "model_name"  : best_clf_name,
    "feature_cols": feature_cols,
    "sel_names"   : sel_names,
    "cmdf_metrics": list(CMDF_METRICS),
    "patch_size"  : [16, 16, 16],
    "n_folds"     : N_FOLDS,
    "ig_top_k"    : IG_TOP_K,
    "random_seed" : RANDOM_SEED,
    "results_dirs": {
        "baseline": RESULTS_DIR,
        "patch"   : PATCH_RESULTS_DIR,
        "cmdf"    : CMDF_RESULTS_DIR,
    },
}
with open(os.path.join(MODEL_DIR, "meta.json"), "w") as fh:
    _json.dump(meta, fh, indent=2)

print(f"\n✅ Model artifacts saved to '{MODEL_DIR}/'")
print("   • scaler.pkl    — StandardScaler fit on all training data")
print("   • sel_names.pkl — IG-selected feature names (top-K list)")
print("   • best_clf.pkl  — best-performing classifier refit on all data")
print("   • meta.json     — feature columns, config & results paths")

Loaded: 62 patients × 1212 features
Classes: {np.int64(0): np.int64(10), np.int64(1): np.int64(52)}
Selected 30 features (top-30 by IG)
  → 20 CMDF  |  10 patch-agg

Best model from CV: CMDF_RF (AUC=0.8982 ± 0.1163)

Refitted CMDF_RF on all 62 patients.

✅ Model artifacts saved to '/content/drive/MyDrive/Capstone_paper_001/brain_mri_model/'
   • scaler.pkl    — StandardScaler fit on all training data
   • sel_names.pkl — IG-selected feature names (top-K list)
   • best_clf.pkl  — best-performing classifier refit on all data
   • meta.json     — feature columns, config & results paths


In [ ]:
import glob
import itertools
import json
import os
import tempfile
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd
import SimpleITK as sitk

# ── 0. SUPPRESS WARNINGS ────────────────────────────────────────────────
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ── CONFIG ──────────────────────────────────────────────────────────────
NEW_DATA_ZIP = "/content/validate.zip"
MODEL_DIR = "brain_mri_model"

MODALITY_FILE_MAP = {
    "T1GD": "T1_GD_normalized.nii.gz",
    "T2": "T2_normalized.nii.gz",
}
# ────────────────────────────────────────────────────────────────────────


# ── 1. LOAD MODEL ARTIFACTS ─────────────────────────────────────────────
with open(os.path.join(MODEL_DIR, "meta.json")) as fh:
  meta = json.load(fh)

scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
sel_names = joblib.load(os.path.join(MODEL_DIR, "sel_names.pkl"))
clf = joblib.load(os.path.join(MODEL_DIR, "best_clf.pkl"))

feature_cols = meta["feature_cols"]
sel_idx = [feature_cols.index(n) for n in sel_names if n in feature_cols]
print(f"✅ Model loaded: {meta.get('model_name', 'Classifier')}")
print(f"   Features: {len(feature_cols)} fused → {len(sel_idx)} selected")


# ── 2. HELPER: SAFE RADIOMICS ───────────────────────────────────────────
def _entropy(arr, bins=64):
  counts, _ = np.histogram(arr, bins=bins)
  probs = counts / max(counts.sum(), 1)
  probs = probs[probs > 0]
  return float(-np.sum(probs * np.log2(probs))) if len(probs) > 0 else 0.0


def _uniformity(arr, bins=64):
  counts, _ = np.histogram(arr, bins=bins)
  probs = counts / max(counts.sum(), 1)
  return float(np.sum(probs**2))


def _skewness(arr):
  mu, sd = arr.mean(), arr.std()
  return float(np.mean(((arr - mu) / sd) ** 3)) if sd > 1e-8 else 0.0


def _kurtosis(arr):
  mu, sd = arr.mean(), arr.std()
  return float(np.mean(((arr - mu) / sd) ** 4) - 3) if sd > 1e-8 else 0.0


def _correlation(arr):
  if len(arr) <= 1 or np.std(arr[:-1]) == 0 or np.std(arr[1:]) == 0:
    return 0.0
  with np.errstate(divide="ignore", invalid="ignore"):
    val = np.corrcoef(arr[:-1], arr[1:])[0, 1]
    return float(val) if not np.isnan(val) else 0.0


def extract_radiomics_sitk(image_path: str) -> dict:
  """Return {original_<category>_<feature>: value} for one NIfTI volume."""
  img = sitk.ReadImage(image_path, sitk.sitkFloat32)
  arr = sitk.GetArrayFromImage(img).flatten()
  arr = arr[arr > 0]
  if arr.size == 0:
    raise ValueError("Empty foreground after masking zeros.")

  spacing = np.array(img.GetSpacing())
  size = np.array(img.GetSize(), dtype=float)
  vol_mm3 = float(np.prod(size * spacing))
  surf = float(
      2
      * (size[0] * size[1] + size[1] * size[2] + size[0] * size[2])
      * np.prod(spacing) ** (2 / 3)
  )

  var_val = float(arr.var())
  feats = {
      "firstorder_Energy": float(np.sum(arr**2)),
      "firstorder_TotalEnergy": float(np.sum(arr**2) * np.prod(spacing)),
      "firstorder_Entropy": _entropy(arr),
      "firstorder_Minimum": float(arr.min()),
      "firstorder_10Percentile": float(np.percentile(arr, 10)),
      "firstorder_90Percentile": float(np.percentile(arr, 90)),
      "firstorder_Maximum": float(arr.max()),
      "firstorder_Mean": float(arr.mean()),
      "firstorder_Median": float(np.median(arr)),
      "firstorder_InterquartileRange": float(
          np.percentile(arr, 75) - np.percentile(arr, 25)
      ),
      "firstorder_Range": float(arr.max() - arr.min()),
      "firstorder_MeanAbsoluteDeviation": float(
          np.mean(np.abs(arr - arr.mean()))
      ),
      "firstorder_Skewness": _skewness(arr),
      "firstorder_Kurtosis": _kurtosis(arr),
      "firstorder_Variance": var_val,
      "firstorder_RootMeanSquared": float(np.sqrt(np.mean(arr**2))),
      "firstorder_Uniformity": _uniformity(arr),
      "shape_VoxelVolume": vol_mm3,
      "shape_MeshVolume": vol_mm3,
      "shape_SurfaceArea": surf,
      "shape_Maximum3DDiameter": float(np.linalg.norm(size * spacing)),
      "shape_Sphericity": float(
          (np.pi ** (1 / 3) * (6 * vol_mm3) ** (2 / 3)) / max(surf, 1e-8)
      ),
      "glcm_JointEnergy": _uniformity(arr),
      "glcm_JointEntropy": _entropy(arr),
      "glcm_Contrast": var_val,
      "glcm_Correlation": _correlation(arr),
      "glcm_Homogeneity": float(1.0 / (1.0 + var_val)),
      "glcm_ClusterShade": _skewness(arr),
      "glcm_ClusterProminence": _kurtosis(arr),
  }
  return {f"original_{k}": v for k, v in feats.items()}


# ── 3. COMPUTE CMDF FEATURES ────────────────────────────────────────────
def compute_cmdf_for_patient(mod_feat_dict: dict, cmdf_metrics=None) -> dict:
  if cmdf_metrics is None:
    cmdf_metrics = meta.get(
        "cmdf_metrics", ["variance", "l2", "cosine", "outlier_deviation"]
    )
  mods = list(mod_feat_dict.keys())
  if len(mods) < 2:
    return {}

  all_feat_names = sorted(
      set.intersection(*[set(mod_feat_dict[m].keys()) for m in mods])
  )
  mat = np.array(
      [[mod_feat_dict[m].get(f, 0.0) for f in all_feat_names] for m in mods],
      dtype=np.float32,
  )

  out = {}
  with np.errstate(divide="ignore", invalid="ignore"):
    for metric in cmdf_metrics:
      if metric == "variance":
        vals = mat.var(axis=0)
      elif metric == "l2":
        vals = np.array([
            np.mean([
                abs(mat[i, fi] - mat[j, fi])
                for i in range(len(mods))
                for j in range(i + 1, len(mods))
            ])
            for fi in range(len(all_feat_names))
        ])
      elif metric == "cosine":
        pairs = list(itertools.combinations(range(len(mods)), 2))
        sims = []
        for i, j in pairs:
          u, v = mat[i], mat[j]
          denom = np.linalg.norm(u) * np.linalg.norm(v)
          sims.append(u * v / max(denom, 1e-8))
        vals = 1.0 - np.mean(sims, axis=0)
      elif metric == "outlier_deviation":
        mean_vec = mat.mean(axis=0)
        vals = np.max(np.abs(mat - mean_vec), axis=0)
      else:
        continue
      vals = np.nan_to_num(vals, nan=0.0)
      for fi, fn in enumerate(all_feat_names):
        out[f"cmdf_{metric}_{fn}"] = float(vals[fi])
  return out


# ── 4. EXTRACT ZIP AND PROCESS PATIENTS ─────────────────────────────────
if not os.path.exists(NEW_DATA_ZIP):
  raise FileNotFoundError(f"Validation zip not found: {NEW_DATA_ZIP}")

tmpdir = tempfile.mkdtemp()
with zipfile.ZipFile(NEW_DATA_ZIP, "r") as zf:
  zf.extractall(tmpdir)

all_dirs = [
    d for d in glob.glob(os.path.join(tmpdir, "**"), recursive=True) if os.path.isdir(d)
]
patient_dirs = [
    d for d in all_dirs if any(f.endswith(".nii.gz") for f in os.listdir(d))
]
print(f"\nPatients found: {[os.path.basename(d) for d in patient_dirs]}")

fused_rows = []
for pdir in patient_dirs:
  pid = os.path.basename(pdir)
  print(f"\n🔍 Processing {pid}")
  mod_feat_dict = {}
  flat_row = {"patient_id": pid}

  for mod_key, fname in MODALITY_FILE_MAP.items():
    img_path = os.path.join(pdir, fname)
    if not os.path.exists(img_path):
      print(f"  ⚠️ Missing {fname} — skipping {mod_key}")
      continue
    try:
      feats = extract_radiomics_sitk(img_path)
      mod_feat_dict[mod_key] = feats
      for feat_name, val in feats.items():
        flat_row[f"{mod_key}_{feat_name}"] = val
      print(f"  ✅ {mod_key}: {len(feats)} features")
    except Exception as e:
      print(f"  ❌ {mod_key} failed: {e}")

  cmdf_feats = compute_cmdf_for_patient(mod_feat_dict)
  flat_row.update(cmdf_feats)
  print(f"  ✅ CMDF: {len(cmdf_feats)} differential features")
  fused_rows.append(flat_row)

df_new = pd.DataFrame(fused_rows).set_index("patient_id")


# ── 5. ALIGN TO TRAINING SCHEMA ─────────────────────────────────────────
df_aligned = df_new.reindex(columns=feature_cols, fill_value=0.0)
X_all = np.nan_to_num(
    df_aligned.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0
)


# ── 6. SCALE AND SELECT ─────────────────────────────────────────────────
X_df = pd.DataFrame(X_all, columns=feature_cols)
X_sc = scaler.transform(X_df)
X_sel = X_sc[:, sel_idx]


# ── 7. PREDICT ──────────────────────────────────────────────────────────
y_pred = clf.predict(X_sel)
y_prob = (
    clf.predict_proba(X_sel)[:, 1]
    if hasattr(clf, "predict_proba")
    else y_pred.astype(float)
)

labels = {0: "Non-codeletion", 1: "1p/19q Codeletion"}
confidence = np.where(y_pred == 1, y_prob, 1 - y_prob)


# ── 8. OUTPUT ───────────────────────────────────────────────────────────
out = pd.DataFrame({
    "patient_id": df_new.index,
    "pred": y_pred,
    "label": [labels[p] for p in y_pred],
    "prob_codeletion": np.round(y_prob, 4),
    "confidence": np.round(confidence, 4),
})

print("\n✅ INFERENCE RESULTS:")
print(out.to_string(index=False))
out.to_csv("prediction.csv", index=False)
print("\nSaved → prediction.csv")

✅ Model loaded: CMDF_RF
   Features: 1212 fused → 30 selected

Patients found: ['validate']

🔍 Processing validate
  ✅ T1GD: 29 features
  ✅ T2: 29 features
  ✅ CMDF: 116 differential features

✅ INFERENCE RESULTS:
patient_id  pred          label  prob_codeletion  confidence
  validate     0 Non-codeletion             0.31        0.69

Saved → prediction.csv


## 9. End-to-End Single-Patient Subtype Detection Pipeline

Given a **single patient folder** placed under `/content/` containing the 4 raw sequences (**T1, T1-Gd, T2, FLAIR**) plus **1 segmentation mask**, this pipeline runs the same preprocessing → tumor-patch extraction → radiomics + CMDF feature computation used at training time, loads the saved model artifacts from Section 8 (`brain_mri_model/`), and outputs a **subtype detection**: Oligodendroglioma (1p/19q-codeleted) vs. Astrocytoma (non-codeleted), with a confidence score.

**Expected folder layout** (filenames are matched flexibly by keyword, so exact naming isn't critical):

```
/content/patient_XYZ/
├── T1.nii.gz
├── T1_Gd.nii.gz          (or T1CE / T1ce / T1_GD ...)
├── T2.nii.gz
├── FLAIR.nii.gz
└── segmentation.nii.gz   (or *_seg* / *mask* / *GlistrBoost* ...)
```

Only **T1-Gd and T2** feed the trained model (matching `config.SEQUENCES`) — T1 and FLAIR are loaded for completeness/visualization but are not required for the prediction itself.

⚠️ This is a research prototype trained on a small dataset — it is **not** a validated diagnostic tool.

### 9.1 Pipeline Module — `pipeline_single_patient.py`

In [ ]:
%%writefile pipeline_single_patient.py
"""
pipeline_single_patient.py -- End-to-end single-patient subtype detection.

Takes a folder containing 4 raw MRI sequences (T1, T1-Gd, T2, FLAIR) + 1
segmentation mask, reproduces the training-time preprocessing / patch /
CMDF feature pipeline on that one patient, then loads the saved model
artifacts (scaler.pkl, sel_names.pkl, best_clf.pkl, meta.json from
brain_mri_model/) to predict 1p/19q codeletion status:

    0 -> Astrocytoma-like   (Non-codeletion)
    1 -> Oligodendroglioma-like (1p/19q Codeletion)
"""
import os
import json
import glob
import joblib
import numpy as np
import pandas as pd
import SimpleITK as sitk
from itertools import combinations

from config import (TARGET_SPACING, N4_ITERATIONS, N4_CONVERGENCE_THRESHOLD,
                    N4_SHRINK_FACTOR, PATCH_SIZE, PATCH_MIN_TUMOR_FRACTION,
                    CMDF_METRICS, UNKNOWN_LABEL)
from mil_aggregation import aggregate_patient_features

SUBTYPE_LABELS = {0: "Astrocytoma-like (Non-codeletion)",
                  1: "Oligodendroglioma-like (1p/19q Codeletion)"}


# ── 1. FILE DISCOVERY ─────────────────────────────────────────────────────
def discover_patient_files(patient_dir):
    """Flexibly locate T1 / T1-Gd / T2 / FLAIR / segmentation files inside a
    single patient folder by filename keyword matching. Returns a dict of
    found paths; missing modalities are simply absent from the dict."""
    exts = (".nii.gz", ".nii")
    all_files = [f for f in os.listdir(patient_dir) if f.lower().endswith(exts)]

    seg_keywords = ("seg", "mask", "label", "glistrboost")
    seg_file = next((f for f in all_files if any(k in f.lower() for k in seg_keywords)), None)

    remaining = [f for f in all_files if f != seg_file]

    def find(keywords, exclude=()):
        for f in remaining:
            fl = f.lower()
            if any(x in fl for x in exclude):
                continue
            if any(k in fl for k in keywords):
                return os.path.join(patient_dir, f)
        return None

    found = {}
    if seg_file:
        found["seg"] = os.path.join(patient_dir, seg_file)
    found["t1Gd"]  = find(["t1gd", "t1_gd", "t1-gd", "t1ce", "t1c"])
    found["t2"]    = find(["t2"], exclude=["flair"])
    found["flair"] = find(["flair"])
    found["t1"]    = find(["t1"], exclude=["t1gd", "t1_gd", "t1-gd", "t1ce", "t1c"])
    return found


# ── 2. PREPROCESSING (N4 bias correction + isotropic resampling) ───────────
def preprocess_volume(path, target_spacing=TARGET_SPACING,
                      n4_iterations=None, convergence=N4_CONVERGENCE_THRESHOLD,
                      shrink=N4_SHRINK_FACTOR):
    if n4_iterations is None:
        n4_iterations = N4_ITERATIONS
    img = sitk.ReadImage(path, sitk.sitkFloat32)
    try:
        mask = sitk.OtsuThreshold(img, 0, 1, 200)
        shrunk_img  = sitk.Shrink(img,  [shrink] * img.GetDimension())
        shrunk_mask = sitk.Shrink(mask, [shrink] * img.GetDimension())
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations(n4_iterations)
        corrector.SetConvergenceThreshold(convergence)
        _ = corrector.Execute(shrunk_img, shrunk_mask)
        log_bias_field = corrector.GetLogBiasFieldAsImage(img)
        img = img / sitk.Exp(log_bias_field)
    except Exception as e:
        print(f"  [WARN] N4 bias correction skipped ({e})")

    resampler = sitk.ResampleImageFilter()
    orig_size    = np.array(img.GetSize())
    orig_spacing = np.array(img.GetSpacing())
    new_size = np.maximum(
        (orig_size * (orig_spacing / np.array(target_spacing))).round().astype(int), 1
    ).tolist()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetInterpolator(sitk.sitkBSpline)
    return resampler.Execute(img)


def resample_like(path, reference_img, is_mask=True):
    src = sitk.ReadImage(path, sitk.sitkUInt8 if is_mask else sitk.sitkFloat32)
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference_img)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkBSpline)
    return resampler.Execute(src)


# ── 3. TUMOR PATCH GRID EXTRACTION ──────────────────────────────────────────
def extract_tumor_patches(t1gd_arr, t2_arr, seg_arr,
                          patch_size=PATCH_SIZE, min_frac=PATCH_MIN_TUMOR_FRACTION):
    pz, py, px = patch_size
    dz, dy, dx = seg_arr.shape
    patches = []
    for z in range(0, max(dz - pz + 1, 1), pz):
        for y in range(0, max(dy - py + 1, 1), py):
            for x in range(0, max(dx - px + 1, 1), px):
                seg_patch = seg_arr[z:z+pz, y:y+py, x:x+px]
                if seg_patch.size == 0:
                    continue
                tumor_frac = float((seg_patch > 0).mean())
                if tumor_frac >= min_frac:
                    patches.append({
                        "bbox": (z, y, x),
                        "t1Gd": t1gd_arr[z:z+pz, y:y+py, x:x+px],
                        "t2":   t2_arr[z:z+pz, y:y+py, x:x+px],
                        "tumor_fraction": tumor_frac,
                    })
    if not patches:
        # fallback: no patch met the threshold -> use the tumor's bounding box as one big patch
        zs, ys, xs = np.where(seg_arr > 0)
        if len(zs) == 0:
            return patches
        z0, z1 = zs.min(), zs.max() + 1
        y0, y1 = ys.min(), ys.max() + 1
        x0, x1 = xs.min(), xs.max() + 1
        patches.append({
            "bbox": (int(z0), int(y0), int(x0)),
            "t1Gd": t1gd_arr[z0:z1, y0:y1, x0:x1],
            "t2":   t2_arr[z0:z1, y0:y1, x0:x1],
            "tumor_fraction": 1.0,
        })
        print("  [INFO] No grid patch met the tumor-fraction threshold -- "
              "falling back to whole tumor bounding box as a single patch.")
    return patches


# ── 4. PER-PATCH RADIOMICS-STYLE FEATURES ───────────────────────────────────
def _entropy(a, bins=32):
    c, _ = np.histogram(a, bins=bins)
    p = c / max(c.sum(), 1); p = p[p > 0]
    return float(-np.sum(p * np.log2(p))) if p.size else 0.0

def _uniformity(a, bins=32):
    c, _ = np.histogram(a, bins=bins)
    p = c / max(c.sum(), 1)
    return float(np.sum(p ** 2))

def _skewness(a):
    mu, sd = a.mean(), a.std()
    return float(np.mean(((a - mu) / sd) ** 3)) if sd > 1e-8 else 0.0

def _kurtosis(a):
    mu, sd = a.mean(), a.std()
    return float(np.mean(((a - mu) / sd) ** 4) - 3) if sd > 1e-8 else 0.0

def patch_firstorder_features(arr):
    a = arr.flatten().astype(np.float64)
    if a.size == 0:
        a = np.zeros(1)
    return {
        "firstorder_Mean"      : float(a.mean()),
        "firstorder_Median"    : float(np.median(a)),
        "firstorder_Std"       : float(a.std()),
        "firstorder_Variance"  : float(a.var()),
        "firstorder_Minimum"   : float(a.min()),
        "firstorder_Maximum"   : float(a.max()),
        "firstorder_Range"     : float(a.max() - a.min()),
        "firstorder_Skewness"  : _skewness(a),
        "firstorder_Kurtosis"  : _kurtosis(a),
        "firstorder_Entropy"   : _entropy(a),
        "firstorder_Uniformity": _uniformity(a),
        "firstorder_Energy"    : float(np.sum(a ** 2)),
        "glcm_Contrast"        : float(a.var()),
        "glcm_Homogeneity"     : float(1.0 / (1.0 + a.var())),
    }


def build_patch_dataframe(patches, patient_id):
    rows, t1_list, t2_list = [], [], []
    for i, p in enumerate(patches):
        t1_feats = patch_firstorder_features(p["t1Gd"])
        t2_feats = patch_firstorder_features(p["t2"])
        t1_list.append(t1_feats); t2_list.append(t2_feats)
        row = {"patient_id": patient_id, "patch_idx": i, "label": UNKNOWN_LABEL}
        row.update({f"T1GD_{k}": v for k, v in t1_feats.items()})
        row.update({f"T2_{k}": v for k, v in t2_feats.items()})
        rows.append(row)
    patch_df = pd.DataFrame(rows)
    t1_mean = {k: float(np.mean([d[k] for d in t1_list])) for k in t1_list[0]}
    t2_mean = {k: float(np.mean([d[k] for d in t2_list])) for k in t2_list[0]}
    return patch_df, {"T1GD": t1_mean, "T2": t2_mean}


# ── 5. CROSS-MODAL DIFFERENTIAL FEATURES (CMDF) ─────────────────────────────
def compute_cmdf_for_patient(mod_feat_dict, cmdf_metrics=None):
    if cmdf_metrics is None:
        cmdf_metrics = CMDF_METRICS
    mods = list(mod_feat_dict.keys())
    if len(mods) < 2:
        return {}
    names = sorted(set.intersection(*[set(mod_feat_dict[m].keys()) for m in mods]))
    mat = np.array([[mod_feat_dict[m].get(f, 0.0) for f in names] for m in mods], dtype=np.float32)

    out = {}
    for metric in cmdf_metrics:
        if metric == "variance":
            vals = mat.var(axis=0)
        elif metric == "l2":
            vals = np.array([
                np.mean([abs(mat[i, fi] - mat[j, fi])
                         for i in range(len(mods)) for j in range(i + 1, len(mods))])
                for fi in range(len(names))
            ])
        elif metric == "cosine":
            pairs = list(combinations(range(len(mods)), 2))
            sims = []
            for i, j in pairs:
                u, v = mat[i], mat[j]
                denom = np.linalg.norm(u) * np.linalg.norm(v)
                sims.append(u * v / max(denom, 1e-8))
            vals = 1.0 - np.mean(sims, axis=0)
        elif metric == "outlier_deviation":
            mean_vec = mat.mean(axis=0)
            vals = np.max(np.abs(mat - mean_vec), axis=0)
        else:
            continue
        for fi, fn in enumerate(names):
            out[f"cmdf_{metric}_{fn}"] = float(vals[fi])
    return out


# ── 6. DETECTION LOGIC (load model, align features, predict) ───────────────
def _load_model_artifacts(model_dir):
    with open(os.path.join(model_dir, "meta.json")) as fh:
        meta = json.load(fh)
    scaler    = joblib.load(os.path.join(model_dir, "scaler.pkl"))
    sel_names = joblib.load(os.path.join(model_dir, "sel_names.pkl"))
    clf       = joblib.load(os.path.join(model_dir, "best_clf.pkl"))
    return meta, scaler, sel_names, clf


def detect_subtype(fused_row, model_dir="brain_mri_model"):
    """Align the single-patient fused feature row to the trained model's
    schema, scale, select, and predict. Returns a result dict."""
    if not os.path.exists(os.path.join(model_dir, "meta.json")):
        print(f"  [WARN] No trained model found in '{model_dir}/' -- "
              "run Section 8 (Model Export) first. Skipping prediction.")
        return None

    meta, scaler, sel_names, clf = _load_model_artifacts(model_dir)
    feature_cols = meta["feature_cols"]
    sel_idx = [feature_cols.index(n) for n in sel_names]

    X = np.array([[fused_row.get(c, 0.0) for c in feature_cols]], dtype=np.float32)
    X = np.nan_to_num(X)
    X_sc  = scaler.transform(X)
    X_sel = X_sc[:, sel_idx]

    y_pred = int(clf.predict(X_sel)[0])
    y_prob = float(clf.predict_proba(X_sel)[0, 1]) if hasattr(clf, "predict_proba") else float(y_pred)
    confidence = y_prob if y_pred == 1 else 1 - y_prob

    return {
        "model_name"     : meta["model_name"],
        "pred"           : y_pred,
        "subtype"        : SUBTYPE_LABELS[y_pred],
        "prob_codeletion": round(y_prob, 4),
        "confidence"     : round(confidence, 4),
    }


# ── 7. MAIN ORCHESTRATOR ─────────────────────────────────────────────────
def run_single_patient_pipeline(patient_dir, model_dir="brain_mri_model", patient_id=None):
    patient_id = patient_id or os.path.basename(os.path.normpath(patient_dir))
    print(f"\n{'='*70}\n  END-TO-END SUBTYPE DETECTION -- {patient_id}\n{'='*70}")

    files = discover_patient_files(patient_dir)
    print("Discovered files:")
    for k in ["t1", "t1Gd", "t2", "flair", "seg"]:
        print(f"  {k:>6s}: {os.path.basename(files[k]) if files.get(k) else '(not found)'}")

    if not files.get("t1Gd") or not files.get("t2") or not files.get("seg"):
        raise FileNotFoundError(
            "T1-Gd, T2 and a segmentation mask are all required "
            "(these are the sequences the trained model actually uses).")

    print("\nPreprocessing (N4 bias correction + resample to "
          f"{TARGET_SPACING} mm isotropic)...")
    t1gd_img = preprocess_volume(files["t1Gd"])
    t2_img   = resample_like(files["t2"], t1gd_img, is_mask=False)
    seg_img  = resample_like(files["seg"], t1gd_img, is_mask=True)

    t1gd_arr = sitk.GetArrayFromImage(t1gd_img)
    t2_arr   = sitk.GetArrayFromImage(t2_img)
    seg_arr  = sitk.GetArrayFromImage(seg_img)
    print(f"  Resampled volume shape: {t1gd_arr.shape}, "
          f"tumor voxels: {int((seg_arr > 0).sum())}")

    print("\nExtracting tumor patches "
          f"({PATCH_SIZE}, min tumor fraction={PATCH_MIN_TUMOR_FRACTION})...")
    patches = extract_tumor_patches(t1gd_arr, t2_arr, seg_arr)
    if not patches:
        raise ValueError("No tumor voxels found in the segmentation mask -- cannot proceed.")
    print(f"  Retained {len(patches)} patches")

    patch_df, mod_feat_dict = build_patch_dataframe(patches, patient_id)
    agg_df = aggregate_patient_features(patch_df)
    cmdf_feats = compute_cmdf_for_patient(mod_feat_dict)
    print(f"  Patch-aggregated features: {agg_df.shape[1] - 3}  |  CMDF features: {len(cmdf_feats)}")

    meta_cols = ["patient_id", "label", "n_patches"]
    fused_row = {c: agg_df.iloc[0][c] for c in agg_df.columns if c not in meta_cols}
    fused_row.update(cmdf_feats)

    print("\nRunning detection logic against saved model...")
    prediction = detect_subtype(fused_row, model_dir=model_dir)

    result = {
        "patient_id": patient_id,
        "n_patches": len(patches),
        "tumor_voxels": int((seg_arr > 0).sum()),
        "volumes": {"t1Gd": t1gd_arr, "t2": t2_arr, "seg": seg_arr},
        "prediction": prediction,
    }

    if prediction:
        print(f"\n{'-'*70}")
        print(f"  PREDICTED SUBTYPE : {prediction['subtype']}")
        print(f"  Model used        : {prediction['model_name']}")
        print(f"  P(1p/19q codel.)  : {prediction['prob_codeletion']:.2%}")
        print(f"  Confidence        : {prediction['confidence']:.2%}")
        print(f"{'-'*70}")
        print("  Detection logic: predicts class 1 (Oligodendroglioma-like) when "
              "P(1p/19q codeletion) >= 0.5, else class 0 (Astrocytoma-like); "
              "confidence = probability of the predicted class.")
        print("  ⚠️  Research prototype -- not a validated clinical diagnostic tool.")

    return result


Writing pipeline_single_patient.py


### 9.2 Self-Test — Synthetic Patient Folder (proves the pipeline runs end-to-end)

Builds a throw-away synthetic patient folder (random NIfTI volumes + a synthetic tumor mask) under `/content/`, then runs the full pipeline on it. This validates every step — file discovery, N4 + resampling, patch extraction, radiomics/CMDF feature computation, and (if `brain_mri_model/` already exists from Section 8) the final prediction — without needing real patient data.

In [ ]:
import os, shutil
import numpy as np
import SimpleITK as sitk
from pipeline_single_patient import run_single_patient_pipeline

def _make_synthetic_patient(root="/content/synthetic_test_patient", size=(48, 48, 48), seed=0):
    rng = np.random.default_rng(seed)
    if os.path.exists(root):
        shutil.rmtree(root)
    os.makedirs(root, exist_ok=True)

    def save(arr, name, spacing=(1.0, 1.0, 1.0)):
        img = sitk.GetImageFromArray(arr.astype(np.float32))
        img.SetSpacing(spacing)
        sitk.WriteImage(img, os.path.join(root, name))

    base = rng.normal(500, 80, size=size)
    save(base + rng.normal(0, 20, size=size), "T1.nii.gz")
    save(base + rng.normal(0, 20, size=size), "T1_Gd.nii.gz")
    save(base + rng.normal(0, 20, size=size), "T2.nii.gz")
    save(base + rng.normal(0, 20, size=size), "FLAIR.nii.gz")

    seg = np.zeros(size, dtype=np.uint8)
    c = np.array(size) // 2
    r = min(size) // 5
    zz, yy, xx = np.ogrid[:size[0], :size[1], :size[2]]
    sphere = (zz - c[0])**2 + (yy - c[1])**2 + (xx - c[2])**2 <= r**2
    seg[sphere] = 1
    save(seg, "segmentation.nii.gz")

    return root

demo_patient_dir = _make_synthetic_patient()
print(f"Synthetic patient folder created at: {demo_patient_dir}")

selftest_result = run_single_patient_pipeline(demo_patient_dir, model_dir="brain_mri_model")

Synthetic patient folder created at: /content/synthetic_test_patient

  END-TO-END SUBTYPE DETECTION -- synthetic_test_patient
Discovered files:
      t1: T1.nii.gz
    t1Gd: T1_Gd.nii.gz
      t2: T2.nii.gz
   flair: FLAIR.nii.gz
     seg: segmentation.nii.gz

Preprocessing (N4 bias correction + resample to (1.0, 1.0, 1.0) mm isotropic)...
  Resampled volume shape: (48, 48, 48), tumor voxels: 3071

Extracting tumor patches ((16, 16, 16), min tumor fraction=0.5)...
  Retained 1 patches
  Aggregated 1 patches -> 1 patients, 140 features
  Patch-aggregated features: 140  |  CMDF features: 56

Running detection logic against saved model...

----------------------------------------------------------------------
  PREDICTED SUBTYPE : Astrocytoma-like (Non-codeletion)
  Model used        : CMDF_RF
  P(1p/19q codel.)  : 31.00%
  Confidence        : 69.00%
----------------------------------------------------------------------
  Detection logic: predicts class 1 (Oligodendroglioma-like) when P(

### 9.3 Run on a Real Patient Folder

In [ ]:
# Point this at a real patient folder placed under /content/, containing
# T1, T1-Gd, T2, FLAIR and a segmentation mask (see layout in the intro above).
PATIENT_FOLDER = "/content/drive/My Drive/Patient_001"   # <-- change this
# MODEL_DIR      = "brain_mri_model"
MODEL_DIR = "/content/drive/MyDrive/Capstone_paper_001/brain_mri_model"

if os.path.isdir(PATIENT_FOLDER):
    real_result = run_single_patient_pipeline(PATIENT_FOLDER, model_dir=MODEL_DIR)
else:
    print(f"'{PATIENT_FOLDER}' not found yet -- update PATIENT_FOLDER above to point "
          "at a real patient folder, then re-run this cell.")
    real_result = None


  END-TO-END SUBTYPE DETECTION -- Patient_001
Discovered files:
      t1: T1.nii.gz
    t1Gd: T1_GD.nii.gz
      t2: T2.nii.gz
   flair: FLAIR.nii.gz
     seg: segmentation.nii.gz

Preprocessing (N4 bias correction + resample to (1.0, 1.0, 1.0) mm isotropic)...
  Resampled volume shape: (155, 240, 240), tumor voxels: 150416

Extracting tumor patches ((16, 16, 16), min tumor fraction=0.5)...
  Retained 30 patches
  Aggregated 30 patches -> 1 patients, 140 features
  Patch-aggregated features: 140  |  CMDF features: 56

Running detection logic against saved model...

----------------------------------------------------------------------
  PREDICTED SUBTYPE : Astrocytoma-like (Non-codeletion)
  Model used        : CMDF_RF
  P(1p/19q codel.)  : 31.00%
  Confidence        : 69.00%
----------------------------------------------------------------------
  Detection logic: predicts class 1 (Oligodendroglioma-like) when P(1p/19q codeletion) >= 0.5, else class 0 (Astrocytoma-like); confidence = 